# Task 1: Dataset Construction
## BioBERT Pressure Ulcer QA System
## 7146COMP Advanced Topics in Deep Learning

---

## Overview

This notebook implements the dataset construction pipeline for a 
BioBERT-based extractive question answering system intended to support 
pressure ulcer training for healthcare practitioners. The pipeline 
constructs a SQuAD 2.0 formatted dataset from authoritative clinical 
documents and targeted peer-reviewed evidence.

The following outputs are produced and saved to disk:

| Output | File | Description |
|--------|------|-------------|
| Corpus | corpus.json | Extracted and cleaned document texts |
| Chunks | chunks.json | Overlapping 400-word text segments |
| QA pairs | qa_pairs.json | Validated answerable QA pairs |
| Full dataset | all_qa_pairs.json | Answerable and unanswerable pairs |
| SQuAD dataset | squad_dataset.json | Train, validation, and test splits |

---

## Task 1 Design Decisions and Justifications

### Decision 1: Clinical Guidelines as Primary Knowledge Source

The selection of source documents is arguably the most consequential 
decision in this task, since the language and content of source 
documents directly determines the types of questions the trained 
model will be capable of answering reliably.

Preliminary prototype evaluation retrieved peer-reviewed research 
abstracts via the PubMed Entrez API. Inspection of the resulting QA 
pairs revealed a persistent and significant mismatch between the 
language of research abstracts and the language of clinical practice. 
Research abstracts characteristically report statistical outcomes, 
study populations, hazard ratios, and methodological detail. While 
this content is academically valid, it does not reflect the questions 
a nurse is likely to ask at the point of care. A nurse managing a 
patient with suspected Category 2 pressure damage is unlikely to ask 
what the odds ratio for ulcer development was in a particular cohort 
study — they are likely to ask what dressing to apply, whether the 
wound requires a tissue viability referral, or how frequently the 
patient should be repositioned. The prototype corpus produced pairs 
that answered the former type of question but not the latter, 
rendering it unsuitable for the intended clinical application.

This implementation addresses that limitation by sourcing documents 
primarily from authoritative clinical guidelines — specifically NICE 
CG179, NICE QS89 and its individual quality statement chapters, the 
National Wound Care Strategy Programme clinical pathway and 
categorisation tool, NHS Trust tissue viability policies from across 
England, Scotland, and Wales, and Wounds UK best practice statements. 
These documents use direct imperative clinical language that closely 
mirrors the queries a practitioner would pose. Additionally, targeted 
peer-reviewed papers are included to provide the evidence base 
underpinning guideline recommendations, covering areas such as risk 
assessment tool validation, repositioning frequency, dressing 
selection, nutritional support, and nurse education. It should be 
noted that the quality and clinical relevance of the resulting dataset 
is nonetheless contingent on the accuracy and currency of these source 
documents, and on the degree to which the downloaded versions reflect 
current NHS practice at the time of deployment.

### Decision 2: Claude Haiku over T5 for QA Generation

This decision concerns the method used to automatically generate 
question-answer pairs from document chunks and has a direct bearing 
on the quality, clinical realism, and extractive validity of every 
training example produced.

Preliminary prototype evaluation employed T5-based question 
generation fine-tuned on SQuAD. This approach yielded a retention 
rate of only 0.75% after automated quality filtering. Manual 
inspection of retained pairs confirmed that several contained 
clinically incorrect or contextually mismatched answers — a 
consequence of T5 having been pre-trained on general-domain text 
corpora that do not reflect clinical terminology or the structure of 
medical guidance documents. T5 frequently generated plausible-sounding 
questions whose answers, when located in the source text, did not 
correspond to the intended clinical fact. This failure mode is 
particularly problematic in a clinical domain where answer precision 
matters.

Claude Haiku is implemented as the QA generation model in this task. 
Claude Haiku processes each passage with full contextual comprehension 
and generates questions that more closely reflect the natural language 
a healthcare practitioner would use at the point of care. Answers are 
required to be exact verbatim spans extracted directly from the source 
passage, and programmatic validation confirms the answer string is 
present character-for-character in the context before the pair is 
accepted. Pairs that fail this grounding check are discarded 
regardless of surface-level plausibility. This approach substantially 
addresses the quality failure observed in prototype evaluation, though 
it should be acknowledged that Claude Haiku may still occasionally 
generate questions that are syntactically valid but clinically 
imprecise, or that over-represent topics that are more verbosely 
described in the source documents. The verbatim grounding requirement 
mitigates answer quality issues but does not eliminate question 
quality variability.

### Decision 3: Numerical Value Preservation in Text Cleaning

This decision concerns the text preprocessing strategy applied to 
extracted document content and affects the clinical completeness 
of every chunk produced for QA generation.

A commonly applied PDF cleaning step removes standalone integers on 
the grounds that isolated numbers in extracted text typically 
represent page numbers, figure labels, or reference indices that 
add noise rather than meaning. However, applying this approach 
uncritically to clinical text is demonstrably harmful. In clinical 
guidelines and NHS policy documents, numerical values frequently 
carry essential clinical meaning that cannot be recovered once 
removed. Repositioning frequency specifications such as every 2 
hours, wound classification categories such as Category 3 pressure 
ulcer, risk score thresholds such as Waterlow score of 15 or above, 
and epidemiological figures such as 700,000 patients affected 
annually are all expressed as numbers. Removing these values produces 
chunks that are clinically incomplete and that may generate answer 
spans missing their most informative elements.

The cleaning function implemented in this pipeline therefore preserves 
all numerical values throughout preprocessing. Removal is restricted 
to confirmed PDF extraction artefacts — specifically Page X of Y 
page numbering patterns and isolated single non-alphabetic characters 
introduced during XML parsing — leaving all clinically meaningful 
numbers intact. A limitation of this approach is that genuinely 
spurious numbers such as figure labels embedded within paragraph 
text may occasionally persist in chunks, potentially introducing 
minor noise into generated QA pairs.

### Decision 4: SQuAD 2.0 Format with Unanswerable Questions

This decision concerns the format of the final dataset and has 
direct implications for the safety behaviour of the system at 
inference time.

SQuAD 1.1 format assumes that every question has an answerable 
span in the provided context. A model trained exclusively on 
SQuAD 1.1 data learns to always return an answer, even when the 
retrieved context does not contain the relevant information. In 
a clinical deployment context where practitioners may act directly 
on system outputs, this behaviour carries patient safety risk — 
a confident but incorrect clinical recommendation is considerably 
more dangerous than an acknowledgement that the system cannot 
reliably answer the question.

SQuAD 2.0 is therefore implemented, introducing unanswerable 
questions alongside answerable ones. Unanswerable pairs are 
constructed using the standard wrong-context pairing method — 
a genuine clinical question is paired with a context passage 
from which its answer cannot be extracted — at a ratio of 
approximately 20%, consistent with the proportions used in the 
original SQuAD 2.0 benchmark. Training on these pairs teaches 
BioBERT to recognise when retrieved context is insufficient 
and to return a structured no-answer response, which at 
inference time is activated by a confidence threshold of 0.50. 
It should be noted that the wrong-context pairing method 
produces unanswerable examples that are somewhat artificial — 
in practice, unanswerable scenarios in clinical deployment may 
arise from more complex retrieval failures than simple 
context mismatch. This represents a limitation of the current 
approach that future work could address through more 
sophisticated unanswerable example generation strategies.

## 1.1 Environment Setup and Library Imports

All libraries required for the dataset construction pipeline are 
imported in this cell. Centralising imports at the outset ensures 
any missing dependencies are identified immediately rather than 
mid-pipeline, and makes the notebook self-contained.

**pypdf** extracts text from PDF clinical guidelines. NHS Trust 
policies, NICE documents, and NWCSP pathway documents are 
distributed as PDFs. pypdf reads each page sequentially and returns 
raw text strings for subsequent cleaning.

**BeautifulSoup** parses HTML content for documents retrieved as 
web pages rather than PDFs. PubMed Central research articles and 
some NICE guidance chapters are HTML-based. BeautifulSoup strips 
HTML markup and navigation elements, returning clean body text.

**anthropic** is the official Anthropic Python client used 
exclusively for QA pair generation in Stage 1.5. The API key is 
loaded from an environment variable and is never hardcoded in 
the notebook.

**tqdm** provides progress bars for long-running operations 
including document download, text extraction, and QA generation. 
Given that the pipeline processes a substantial number of documents 
and may run for an extended period, progress monitoring is 
considered essential for practical use.

A fixed random seed of 42 is applied to ensure the dataset splits 
produced at the end of this notebook are reproducible across runs.

In [3]:
# =============================================================================
# Environment Setup and Library Imports
# All libraries required for the Task 1 pipeline are imported here.
# A fixed random seed ensures reproducibility of dataset splits.
# =============================================================================

import os
import re
import json
import time
import random
import warnings
import numpy as np
from tqdm import tqdm
from pathlib import Path

# PDF text extraction
import pypdf

# HTML text extraction
from bs4 import BeautifulSoup

# Anthropic API for QA pair generation
import anthropic

# Suppress non-critical warnings for clean output
warnings.filterwarnings('ignore')

# Fixed random seed for reproducibility across all random operations
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Task 1: Dataset Construction")
print("All libraries imported successfully.")

Task 1: Dataset Construction
All libraries imported successfully.


## 1.2 Configuration

All pipeline parameters are defined in a single configuration block. 
Centralising settings here means any parameter adjustment — chunk 
size, target pair count, API model — is made in one place only, 
eliminating the risk of inconsistent values across subsequent cells.

The Anthropic API key is loaded exclusively from an environment 
variable set in the terminal before launching Jupyter. This is 
standard security practice ensuring the key never appears in the 
submitted notebook. The correct procedure is:

Windows:    set ANTHROPIC_API_KEY=your-key-here
Linux/Mac:  export ANTHROPIC_API_KEY=your-key-here

Then launch Jupyter from the same terminal session.

**Chunk size** is set to 400 words with 100-word overlap. A 400-word 
chunk approximates BioBERT's 512 WordPiece token limit, ensuring 
that tokenised examples fit within the model's architectural 
constraints without truncation. The 100-word overlap prevents answer 
spans from being split across chunk boundaries — a clinical statement 
that spans the end of one chunk and the beginning of the next will 
be fully captured in at least one chunk.

**Target QA pairs** is set to 6,000 — within the upper range of the 
assessment specification of 3,000 to 8,000 pairs — to maximise 
training data volume while remaining within the available API credit 
budget.

**Unanswerable ratio** of 20% is consistent with the SQuAD 2.0 
benchmark proportion and ensures the model develops appropriately 
calibrated confidence across both answerable and unanswerable 
scenarios.

In [11]:
# =============================================================================
# Configuration
# All pipeline parameters are centralised here.
# Modify values here only — do not hardcode in subsequent cells.
# =============================================================================

# -----------------------------------------------------------------------------
# Folder and file paths
# -----------------------------------------------------------------------------
GUIDELINES_FOLDER = "./knowledge_base"  # Downloaded source documents
CORPUS_FILE        = "./corpus.json"          # Extracted document texts
CHUNKS_FILE        = "./chunks.json"          # Overlapping text chunks
QA_PAIRS_FILE      = "./qa_pairs.json"        # Answerable QA pairs
ALL_QA_FILE        = "./all_qa_pairs.json"    # Answerable + unanswerable
SQUAD_FILE         = "./squad_dataset.json"   # Final SQuAD 2.0 splits

# -----------------------------------------------------------------------------
# Text preprocessing
# -----------------------------------------------------------------------------
CHUNK_SIZE      = 400   # Words per chunk
CHUNK_OVERLAP   = 100   # Overlapping words between consecutive chunks
MIN_CHUNK_WORDS = 50    # Chunks below this threshold are discarded

# Clinical synonym standardisation
# Ensures clinically identical concepts are treated as semantically
# equivalent throughout the corpus
TERM_MAP = {
    "pressure sore":     "pressure ulcer",
    "pressure sores":    "pressure ulcers",
    "bed sore":          "pressure ulcer",
    "bed sores":         "pressure ulcers",
    "decubitus ulcer":   "pressure ulcer",
    "decubitus ulcers":  "pressure ulcers",
    "pressure injury":   "pressure ulcer",
    "pressure injuries": "pressure ulcers",
}

# -----------------------------------------------------------------------------
# QA generation
# -----------------------------------------------------------------------------
ANTHROPIC_MODEL    = "claude-haiku-4-5-20251001"
QA_PAIRS_PER_CHUNK = 3      # Pairs requested per chunk from Claude Haiku
TARGET_QA_PAIRS    = 6000   # Stop generation when this target is reached
MIN_ANSWER_WORDS   = 4      # Minimum words in a valid answer span
MAX_ANSWER_WORDS   = 50     # Maximum words in a valid answer span
UNANSWERABLES_RATIO = 0.20  # 20% unanswerable pairs

# -----------------------------------------------------------------------------
# Dataset splitting
# -----------------------------------------------------------------------------
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# -----------------------------------------------------------------------------
# API key — loaded from environment variable only
# Set before launching Jupyter:
#   Windows:   set ANTHROPIC_API_KEY=my-key-here
#   Linux/Mac: export ANTHROPIC_API_KEY=my-key-here
# -----------------------------------------------------------------------------
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if ANTHROPIC_API_KEY:
    print("Anthropic API key loaded successfully.")
else:
    print("WARNING: ANTHROPIC_API_KEY not set. QA generation will fail.")
    print("Close Jupyter, run: set ANTHROPIC_API_KEY=your-key")
    print("Then relaunch Jupyter from the same terminal.")

print(f"\nConfiguration loaded:")
print(f"  Guidelines folder:  {GUIDELINES_FOLDER}")
print(f"  Chunk size:         {CHUNK_SIZE} words with {CHUNK_OVERLAP} word overlap")
print(f"  QA pairs per chunk: {QA_PAIRS_PER_CHUNK}")
print(f"  Target QA pairs:    {TARGET_QA_PAIRS:,}")
print(f"  Unanswerable ratio: {UNANSWERABLES_RATIO}")
print(f"  Train/Val/Test:     {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")
print(f"  Model:              {ANTHROPIC_MODEL}")

Anthropic API key loaded successfully.

Configuration loaded:
  Guidelines folder:  ./knowledge_base
  Chunk size:         400 words with 100 word overlap
  QA pairs per chunk: 3
  Target QA pairs:    6,000
  Unanswerable ratio: 0.2
  Train/Val/Test:     0.7/0.15/0.15
  Model:              claude-haiku-4-5-20251001


## 1.3 Knowledge Base Construction — Document Download

Clinical guidelines and targeted peer-reviewed papers are downloaded 
programmatically from their authoritative public sources. Embedding 
the complete source list directly in the notebook ensures full 
reproducibility — any researcher can re-run this cell to reconstruct 
the identical knowledge base from scratch.

Documents are organised into four categories:

**NICE National Standards** — NICE CG179 (the primary national 
clinical guideline for pressure ulcer prevention and management) and 
NICE QS89 (the associated quality standard with individual quality 
statement chapters covering risk assessment, skin inspection, 
repositioning, nutrition, and pressure redistribution devices).

**NHS England and NWCSP Documents** — NHS Improvement definition 
and measurement guidance, the National Wound Care Strategy Programme 
clinical pathway and categorisation tool, and supporting NHS England 
operational documents covering reporting, core curriculum, and the 
Stop the Pressure programme.

**NHS Trust Tissue Viability Policies** — Operational pressure ulcer 
prevention and management policies from NHS Trusts across England, 
Scotland, and Wales. These documents contain the direct procedural 
language nurses follow in practice, including care pathways, 
equipment selection guides, aSSKINg care plans, and incident 
reporting requirements.

**Targeted Peer-Reviewed Evidence** — Open-access full-text articles 
retrieved from PubMed Central covering risk assessment tool validation 
(Braden scale, PURPOSE-T), repositioning frequency evidence, dressing 
efficacy systematic reviews, nutritional support evidence, nurse 
education interventions, and support surface comparisons. These papers 
provide the evidence base underpinning the clinical guideline 
recommendations included in the knowledge base.

Documents that cannot be retrieved due to server unavailability, 
authentication requirements, or URL changes are logged and excluded 
from the corpus. The pipeline continues with all successfully 
downloaded documents. A minimum file size threshold of 5KB is 
applied to detect and reject HTML error pages returned in place 
of actual documents.

In [7]:
# =============================================================================
# Knowledge Base Document Download
# Downloads clinical guidelines and research papers from public sources.
# Organised by category for transparency and reproducibility.
# Failed downloads are logged and skipped — pipeline continues.
# Minimum 5KB threshold rejects HTML error pages.
# =============================================================================

import urllib.request
import ssl

# SSL context handles non-standard certificates on some NHS sites
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode   = ssl.CERT_NONE

os.makedirs(GUIDELINES_FOLDER, exist_ok=True)

ALL_SOURCES = [

    # =========================================================================
    # NICE NATIONAL STANDARDS
    # =========================================================================
    {
        "title":    "NICE CG179: Pressure Ulcers Prevention and Management 2014",
        "url":      "https://www.nice.org.uk/guidance/cg179/resources/pressure-ulcers-prevention-and-management-pdf-35109760631749",
        "source":   "NICE_CG179",
        "category": "NICE"
    },
    {
        "title":    "NICE QS89: Pressure Ulcers Quality Standards 2015",
        "url":      "https://www.nice.org.uk/guidance/qs89/resources/pressure-ulcers-pdf-2098916972485",
        "source":   "NICE_QS89",
        "category": "NICE"
    },
    {
        "title":    "NICE QS89 Chapter: Risk Assessment in Hospital",
        "url":      "https://www.nice.org.uk/guidance/qs89/chapter/Quality-statement-1-Pressure-ulcer-risk-assessment-in-hospital",
        "source":   "NICE_QS89_Hospital_Risk",
        "category": "NICE"
    },
    {
        "title":    "NICE QS89 Chapter: Risk Assessment in Community Nursing",
        "url":      "https://www.nice.org.uk/guidance/qs89/chapter/Quality-statement-2-Pressure-ulcer-risk-assessment-by-community-nursing-services",
        "source":   "NICE_QS89_Community",
        "category": "NICE"
    },
    {
        "title":    "NICE QS89 Chapter: Risk Reassessment",
        "url":      "https://www.nice.org.uk/guidance/qs89/chapter/quality-statement-3-pressure-ulcer-risk-reassessment",
        "source":   "NICE_QS89_Reassessment",
        "category": "NICE"
    },
    {
        "title":    "NICE QS89 Chapter: Skin Assessment",
        "url":      "https://www.nice.org.uk/guidance/qs89/chapter/Quality-statement-4-Skin-assessment",
        "source":   "NICE_QS89_SkinAssessment",
        "category": "NICE"
    },
    {
        "title":    "NICE QS89 Chapter: Repositioning Advice",
        "url":      "https://www.nice.org.uk/guidance/qs89/chapter/Quality-statement-5-Advice-on-repositioning",
        "source":   "NICE_QS89_Repositioning",
        "category": "NICE"
    },
    {
        "title":    "NICE QS89 Chapter: Nutrition and Hydration",
        "url":      "https://www.nice.org.uk/guidance/qs89/resources/pressure-ulcers-pdf-2098916972485",
        "source":   "NICE_QS89_Nutrition",
        "category": "NICE"
    },
    {
        "title":    "NICE QS89 Chapter: Information on Preventing Pressure Ulcers",
        "url":      "https://www.nice.org.uk/guidance/qs89/chapter/Quality-statement-7-Information-on-preventing-pressure-ulcers",
        "source":   "NICE_QS89_Info",
        "category": "NICE"
    },
    {
        "title":    "NICE QS89 Chapter: Pressure Redistribution Devices",
        "url":      "https://www.nice.org.uk/guidance/qs89/chapter/Quality-statement-8-Pressure-redistribution-devices",
        "source":   "NICE_QS89_Devices",
        "category": "NICE"
    },
    {
        "title":    "NICE CG179 Chapter: Risk Assessment Recommendations",
        "url":      "https://www.nice.org.uk/guidance/cg179/chapter/1-Recommendations#risk-assessment",
        "source":   "NICE_CG179_RiskAssessment",
        "category": "NICE"
    },
    {
        "title":    "NICE CG179 Chapter: Skin Assessment and Care",
        "url":      "https://www.nice.org.uk/guidance/cg179/chapter/1-Recommendations#skin-assessment-and-care",
        "source":   "NICE_CG179_SkinAssessment",
        "category": "NICE"
    },
    {
        "title":    "NICE CG179 Chapter: Managing Pressure Ulcers",
        "url":      "https://www.nice.org.uk/guidance/cg179/chapter/1-Recommendations#managing-pressure-ulcers",
        "source":   "NICE_CG179_Treatment",
        "category": "NICE"
    },
    {
        "title":    "NICE CG179 Patient Information",
        "url":      "https://www.nice.org.uk/guidance/cg179/informationforpublic",
        "source":   "NICE_CG179_PatientInfo",
        "category": "NICE"
    },

    # =========================================================================
    # NHS IMPROVEMENT AND ENGLAND
    # =========================================================================
    {
        "title":    "NHS Improvement: Pressure Ulcers Revised Definition and Measurement 2018",
        "url":      "https://www.england.nhs.uk/wp-content/uploads/2021/09/NSTPP-summary-recommendations.pdf",
        "source":   "NHS_Improvement_2018",
        "category": "NHS England"
    },
    {
        "title":    "NHS England: Pressure Ulcer Core Curriculum 2018",
        "url":      "https://www.england.nhs.uk/wp-content/uploads/2021/09/Pressure-ulcer-core-curriculum.pdf",
        "source":   "NHS_CoreCurriculum",
        "category": "NHS England"
    },
    {
        "title":    "NHS England: Guidance for Reporting Pressure Ulcers",
        "url":      "https://www.england.nhs.uk/wp-content/uploads/2021/09/Guidance-for-reporting-pressure-ulcers.pdf",
        "source":   "NHS_Reporting_PU",
        "category": "NHS England"
    },
    {
        "title":    "NHS England: Stop the Pressure One Year On",
        "url":      "https://www.england.nhs.uk/wp-content/uploads/2020/08/Stop_the_Pressure_one_year_on.pdf",
        "source":   "NHS_StopPressure_OneYear",
        "category": "NHS England"
    },
    {
        "title":    "NHS England: Stop the Pressure Definition and Measurement Responses",
        "url":      "https://www.england.nhs.uk/wp-content/uploads/2021/09/Stop-the-Pressure-Definition-and-Measurement-summary-of-responses.pdf",
        "source":   "NHS_Definition_Measurement",
        "category": "NHS England"
    },
    {
        "title":    "NHS England: Safeguarding Adults and Pressure Ulcers Protocol",
        "url":      "https://assets.publishing.service.gov.uk/media/5a7f525040f0b62305b89c32/safeguarding-adults-protocol-pressure-ulcers.pdf",
        "source":   "NHS_Safeguarding_PU",
        "category": "NHS England"
    },

    # =========================================================================
    # NATIONAL WOUND CARE STRATEGY PROGRAMME
    # =========================================================================
    {
        "title":    "NWCSP: Pressure Ulcer Categorisation Tool 2024",
        "url":      "https://www.dbth.nhs.uk/wp-content/uploads/2025/03/NWCSP-Pressure-Ulcer-Categorisation-Tool-2024.pdf",
        "source":   "NWCSP_Categorisation_2024",
        "category": "NWCSP"
    },
    {
        "title":    "Oxford Health NWCSP: Pressure Ulcer Clinical Recommendations 2025",
        "url":      "https://www.oxfordhealth.nhs.uk/wp-content/uploads/sites/51/2025/01/NWCSP-PU-Clinical-Recommendations-and-pathway-final-24.10.23.pdf",
        "source":   "Oxford_NWCSP_2025",
        "category": "NWCSP"
    },

    # =========================================================================
    # WOUNDS UK BEST PRACTICE STATEMENTS
    # =========================================================================
    {
        "title":    "Wounds UK: Best Practice Statement Eliminating Pressure Ulcers 2023",
        "url":      "https://wounds-uk.com/wp-content/uploads/2023/02/676ba6d3ad4aac6993fe0d9eb81bf1a9.pdf",
        "source":   "WoundsUK_BPS_PU_2023",
        "category": "Wounds UK"
    },
    {
        "title":    "Wounds UK: Eliminating Avoidable Pressure Ulcers",
        "url":      "https://wounds-uk.com/wp-content/uploads/2023/02/content_11700.pdf",
        "source":   "WoundsUK_Eliminating",
        "category": "Wounds UK"
    },
    {
        "title":    "Wounds UK: NWCSP Update Pressure Ulcer Prevention 2023",
        "url":      "https://wounds-uk.com/wp-content/uploads/2023/09/WUK_2023_19_3_78_81_NWCSP.pdf",
        "source":   "WoundsUK_NWCSP_2023",
        "category": "Wounds UK"
    },
    {
        "title":    "Wounds UK: Embracing Change Pressure Ulcer Care 2025",
        "url":      "https://wounds-uk.com/wp-content/uploads/2025/09/WUK_21-3_34-39_McGrath_FINAL_Updated.pdf",
        "source":   "WoundsUK_Change_2025",
        "category": "Wounds UK"
    },
    {
        "title":    "Wounds UK: Stop the Pressure Programme Achievements",
        "url":      "https://wounds-uk.com/wp-content/uploads/2023/02/946fc7e2d29067bfb63301be6bba5ed5.pdf",
        "source":   "WoundsUK_StopPressure",
        "category": "Wounds UK"
    },
    {
        "title":    "Wounds UK: Lower Leg Wound Prevention Best Practice 2024",
        "url":      "https://wounds-uk.com/wp-content/uploads/2024/08/LR24_BPS_Prevention_WUK-web.pdf",
        "source":   "WoundsUK_LowerLeg_2024",
        "category": "Wounds UK"
    },

    # =========================================================================
    # NHS TRUST TISSUE VIABILITY POLICIES — ENGLAND
    # =========================================================================
    {
        "title":    "Leicestershire Partnership NHS Trust: Pressure Ulcer Prevention Policy 2024",
        "url":      "https://www.leicspart.nhs.uk/wp-content/uploads/2024/07/Pressure-Ulcer-Prevention-Policy-exp-July-2027-v14.1.pdf",
        "source":   "LPT_NHS_2024",
        "category": "NHS Trust"
    },
    {
        "title":    "Birmingham Solihull Mental Health NHS Trust: Pressure Ulcer Management Policy 2024",
        "url":      "https://www.bsmhft.nhs.uk/wp-content/uploads/2024/01/Pressure-Ulcer-Management-and-Prevention.pdf",
        "source":   "BSMHFT_NHS_2024",
        "category": "NHS Trust"
    },
    {
        "title":    "TEWV NHS Trust: Pressure Ulcer Assessment Prevention and Management",
        "url":      "https://www.tewv.nhs.uk/wp-content/uploads/2021/11/Pressure-Ulcers-Assessment-Prevention-and-Management-of-Procedure.pdf",
        "source":   "TEWV_NHS",
        "category": "NHS Trust"
    },
    {
        "title":    "TEWV NHS: Pressure Ulcers Procedure Document",
        "url":      "https://www.tewv.nhs.uk/wp-content/uploads/2021/12/Pressure-Ulcers.pdf",
        "source":   "TEWV_NHS_PU_Doc",
        "category": "NHS Trust"
    },
    {
        "title":    "Doncaster Bassetlaw NHS Trust: Pressure Ulcer Clinical Pathway 2024",
        "url":      "https://www.dbth.nhs.uk/wp-content/uploads/2024/07/Pressure-Ulcer-Clinical-Pathway-Secondary-Care-2024-2027.pdf",
        "source":   "DBTH_NHS_2024",
        "category": "NHS Trust"
    },
    {
        "title":    "DBTH: aSSKINg Red Care Plan 2024",
        "url":      "https://www.dbth.nhs.uk/wp-content/uploads/2024/04/aSSKINg-Red-Care-Plan.pdf",
        "source":   "DBTH_aSSKINg_Red_2024",
        "category": "NHS Trust"
    },
    {
        "title":    "DBTH: aSSKINg Amber Care Plan 2024",
        "url":      "https://www.dbth.nhs.uk/wp-content/uploads/2024/04/aSSKINg-Amber-Care-Plan.pdf",
        "source":   "DBTH_aSSKINg_Amber_2024",
        "category": "NHS Trust"
    },
    {
        "title":    "DBTH: aSSKINg Green Care Plan 2024",
        "url":      "https://www.dbth.nhs.uk/wp-content/uploads/2024/04/aSSKINg-Green-Care-Plan.pdf",
        "source":   "DBTH_aSSKINg_Green_2024",
        "category": "NHS Trust"
    },
    {
        "title":    "DBTH: Pressure Ulcer Categorisation Tool 2024",
        "url":      "https://www.dbth.nhs.uk/wp-content/uploads/2024/07/Pressure-Ulcer-Categorisation-Tool.pdf",
        "source":   "DBTH_PU_Categorisation_2024",
        "category": "NHS Trust"
    },
    {
        "title":    "DBTH: Medical Device Related Pressure Ulcer Prevention 2024",
        "url":      "https://www.dbth.nhs.uk/wp-content/uploads/2024/04/Prevention-of-Medical-Device-Related-Pressure-Ulcers-MDRPU-guidance-2024-2027.pdf",
        "source":   "DBTH_MDRPU_2024",
        "category": "NHS Trust"
    },
    {
        "title":    "DBTH: Pressure Ulcer Product Selection 2024",
        "url":      "https://www.dbth.nhs.uk/wp-content/uploads/2024/01/Pressure-ulcer-product-selection-v2-2024.pdf",
        "source":   "DBTH_Product_Selection_2024",
        "category": "NHS Trust"
    },
    {
        "title":    "DBTH: Wound Care Formulary and Pathway 2024",
        "url":      "https://www.dbth.nhs.uk/wp-content/uploads/2024/10/Wound-Care-Formulary-Navigation-Pathway.pdf",
        "source":   "DBTH_Wound_Formulary_2024",
        "category": "NHS Trust"
    },
    {
        "title":    "DBTH: Preventing Pressure Ulcers Patient Information 2024",
        "url":      "https://www.dbth.nhs.uk/wp-content/uploads/2024/07/WPR50010-Preventing-pressure-ulcers.pdf",
        "source":   "DBTH_Patient_Info_2024",
        "category": "NHS Trust"
    },
    {
        "title":    "DBTH: Foot Ulcer Pathway Primary Care 2024",
        "url":      "https://www.dbth.nhs.uk/wp-content/uploads/2024/01/Foot-Ulcer-Pathway-Primary-Care-2024.pdf",
        "source":   "DBTH_Foot_Ulcer_2024",
        "category": "NHS Trust"
    },
    {
        "title":    "Wrightington Wigan Leigh NHS Trust: Pressure Ulcer SOP 2025",
        "url":      "https://www.wwl.nhs.uk/media/FOI/202425/March%202025/10489%20-%20TW21-101%20SOP%201%20Pressure%20Ulcer%20Prevention%20and%20Management%20Procedure_.pdf",
        "source":   "WWL_NHS_SOP_2025",
        "category": "NHS Trust"
    },
    {
        "title":    "Wrightington Wigan Leigh NHS Trust: Pressure Ulcer Prevention Policy 2025",
        "url":      "https://www.wwl.nhs.uk/media/FOI/202526/06%20-%20September%202025/10977%20-%20Pressure%20Ulcer%20Prevention%20and%20Management%20Policy.pdf",
        "source":   "WWL_NHS_Policy_2025",
        "category": "NHS Trust"
    },
    {
        "title":    "CNTW NHS Trust: Pressure Ulcers Patient Information 2024",
        "url":      "https://www.cntw.nhs.uk/wp-content/uploads/2024/11/Pressure-ulcers-2024.pdf",
        "source":   "CNTW_NHS_2024",
        "category": "NHS Trust"
    },
    {
        "title":    "ELFT NHS Trust: Wound Management Clinical Practice Guidelines",
        "url":      "https://www.elft.nhs.uk/sites/default/files/2025-08/foi_da6196_-_appendix_1_-_wound_management_guidelines_7.3.pdf",
        "source":   "ELFT_NHS_WoundMgt",
        "category": "NHS Trust"
    },
    {
        "title":    "BSW ICB NHS: Pressure Ulcers Quality Standards",
        "url":      "https://bsw.icb.nhs.uk/wp-content/uploads/sites/6/2022/06/Pressure-ulcers-quality-standards.pdf",
        "source":   "BSW_ICB_NHS",
        "category": "NHS Trust"
    },
    {
        "title":    "Liverpool Heart and Chest NHS: Mandatory Training Pressure Ulcer Prevention 2021",
        "url":      "https://www.lhch.nhs.uk/resources/download/lhch-64ba5b400a4337.71744981",
        "source":   "LHCH_Training_2021",
        "category": "NHS Trust"
    },
    {
        "title":    "Guy's and St Thomas NHS: Pressure Ulcer Prevention Policy",
        "url":      "https://www.guysandstthomas.nhs.uk/media/documents/pressure-ulcer-prevention.pdf",
        "source":   "GSTT_NHS_PU",
        "category": "NHS Trust"
    },
    {
        "title":    "University College London Hospitals NHS: Pressure Ulcer Policy",
        "url":      "https://www.uclh.nhs.uk/media/documents/pressure-ulcer-prevention-policy.pdf",
        "source":   "UCLH_NHS_PU",
        "category": "NHS Trust"
    },
    {
        "title":    "Leeds Teaching Hospitals NHS: Pressure Ulcer Patient Information",
        "url":      "https://www.leedsth.nhs.uk/patients/resources/bed-sores-pressure-sores-and-pressure-ulcers/",
        "source":   "Leeds_NHS_Patient_Info",
        "category": "NHS Trust"
    },
    {
        "title":    "Cambridgeshire NHS Trust: Wound Care Guidelines",
        "url":      "https://www.cpft.nhs.uk/download/g09-wound-care-guidelines.pdf?ver=19602",
        "source":   "CPFT_NHS_WoundCare",
        "category": "NHS Trust"
    },
    {
        "title":    "NHS Wales: All Wales Pressure Ulcer Prevention Guidelines",
        "url":      "https://phw.nhs.wales/files/wound-management/pressure-ulcer-prevention-and-management-all-wales-guidance/",
        "source":   "NHS_Wales_PU",
        "category": "NHS Trust"
    },

    # =========================================================================
    # NHS SCOTLAND POLICIES
    # =========================================================================
    {
        "title":    "NHS Greater Glasgow Clyde: Pressure Ulcer Prevention and Management Policy 2024",
        "url":      "https://www.rightdecisions.scot.nhs.uk/media/abmleozl/july-2024-nhsggc-pressure-ulcer-policy-with-podiatry-changes-dec-2024.pdf",
        "source":   "NHS_GGC_Scotland_2024",
        "category": "NHS Scotland"
    },
    {
        "title":    "NHS Lothian: Prevention and Management of Pressure Ulcers Policy",
        "url":      "https://policyonline.nhslothian.scot/wp-content/uploads/2023/03/Prevention_and_Management_of_Pressure_Ulcers_Policy.pdf",
        "source":   "NHS_Lothian",
        "category": "NHS Scotland"
    },
    {
        "title":    "NHS Grampian: Pressure Ulcer Prevention and Management Policy 2024",
        "url":      "https://www.hi-netgrampian.scot.nhs.uk/wp-content/uploads/2024/04/Pressure-Ulcer-Prevention-and-Management-Policy.pdf",
        "source":   "NHS_Grampian_2024",
        "category": "NHS Scotland"
    },
    {
        "title":    "NHS Highland: Pressure Ulcer Prevention and Treatment Guideline",
        "url":      "https://www.nhshighland.scot.nhs.uk/media/documents/pressure-ulcer-guideline.pdf",
        "source":   "NHS_Highland_PU",
        "category": "NHS Scotland"
    },
    {
        "title":    "NHS Tayside: Pressure Ulcer Prevention and Management Guideline",
        "url":      "https://www.nhstayside.scot.nhs.uk/media/documents/pressure-ulcer-prevention-management.pdf",
        "source":   "NHS_Tayside_PU",
        "category": "NHS Scotland"
    },

    # =========================================================================
    # INTERNATIONAL GUIDELINES
    # =========================================================================
    {
        "title":    "NPUAP EPUAP PPPIA: Prevention and Treatment Clinical Practice Guideline 2014",
        "url":      "https://www.andeal.org/files/files/WoundCare/NPUAP-EPUAP-PPPIA%20CPG%202014.pdf",
        "source":   "NPUAP_EPUAP_CPG_2014",
        "category": "International"
    },
    {
        "title":    "WHS Guidelines for Treatment of Pressure Ulcers 2023 Update",
        "url":      "https://docred-strapi-cms-prod.s3.sa-east-1.amazonaws.com/WHS_2023_tratamiento_de_ulceras_por_presion_cb2813d339.pdf",
        "source":   "WHS_PU_Treatment_2023",
        "category": "International"
    },

    # =========================================================================
    # TARGETED PEER-REVIEWED EVIDENCE
    # Open access full text from PubMed Central and other sources
    # Selected to cover the evidence base for guideline recommendations
    # =========================================================================
    {
        "title":    "Predictive Validity Braden Scale Meta-Analysis 2021",
        "url":      "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8363405/",
        "source":   "Braden_Scale_MetaAnalysis_2021",
        "category": "Research"
    },
    {
        "title":    "Oxford Health NHS: Braden Scale Teaching Resource",
        "url":      "https://www.oxfordhealth.nhs.uk/wp-content/uploads/2015/08/Braden-teaching.pdf",
        "source":   "Braden_Teaching_OxfordHealth",
        "category": "Research"
    },
    {
        "title":    "Hartford Institute: Braden Scale Predicting Pressure Injury Risk Guide",
        "url":      "https://hign.org/sites/default/files/2020-06/Try_This_General_Assessment_5.pdf",
        "source":   "Hartford_Braden_Guide",
        "category": "Research"
    },
    {
        "title":    "Repositioning for Pressure Injury Prevention Cochrane Review 2021",
        "url":      "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8256643/",
        "source":   "Repositioning_Cochrane_2021",
        "category": "Research"
    },
    {
        "title":    "Patient Repositioning During Hospitalisation Pressure Ulcer Review",
        "url":      "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC11290892/",
        "source":   "Repositioning_Narrative_Review",
        "category": "Research"
    },
    {
        "title":    "WHS Treatment Guidelines Evidence Review 2023",
        "url":      "https://pmc.ncbi.nlm.nih.gov/articles/PMC11403384/",
        "source":   "WHS_PU_Treatment_Guidelines_2023",
        "category": "Research"
    },
    {
        "title":    "Nurses Knowledge Attitudes Practices Systematic Review",
        "url":      "https://pmc.ncbi.nlm.nih.gov/articles/PMC8249000/",
        "source":   "Nurses_Knowledge_Systematic_Review",
        "category": "Research"
    },
    {
        "title":    "Nutrition Pressure Ulcer Prevention and Treatment Evidence",
        "url":      "https://pmc.ncbi.nlm.nih.gov/articles/PMC7918275/",
        "source":   "Nutrition_PU_Evidence_Review",
        "category": "Research"
    },
    {
        "title":    "Foam Dressings Pressure Ulcer Prevention Systematic Review",
        "url":      "https://pmc.ncbi.nlm.nih.gov/articles/PMC6494411/",
        "source":   "Foam_Dressings_Systematic_Review",
        "category": "Research"
    },
    {
        "title":    "Wound Dressings Treating Pressure Ulcers Evidence Review",
        "url":      "https://pmc.ncbi.nlm.nih.gov/articles/PMC8094168/",
        "source":   "Wound_Dressings_Evidence_Review",
        "category": "Research"
    },
    {
        "title":    "Pressure Redistribution Support Surfaces Systematic Review",
        "url":      "https://pmc.ncbi.nlm.nih.gov/articles/PMC6492148/",
        "source":   "Support_Surfaces_Systematic_Review",
        "category": "Research"
    },
    {
        "title":    "Deep Tissue Pressure Injury Identification and Management",
        "url":      "https://pmc.ncbi.nlm.nih.gov/articles/PMC7196615/",
        "source":   "DTPI_Identification_Management",
        "category": "Research"
    },
    {
        "title":    "PURPOSE-T Pressure Ulcer Risk Assessment Tool Clinical Evaluation",
        "url":      "https://pmc.ncbi.nlm.nih.gov/articles/PMC5810903/",
        "source":   "PURPOSE_T_Clinical_Evaluation",
        "category": "Research"
    },
    {
        "title":    "Negative Pressure Wound Therapy Pressure Ulcers Systematic Review",
        "url":      "https://pmc.ncbi.nlm.nih.gov/articles/PMC6492463/",
        "source":   "NPWT_PU_Systematic_Review",
        "category": "Research"
    },
    {
        "title":    "Pressure Ulcer Prevention Education Nurses Evidence Review",
        "url":      "https://pmc.ncbi.nlm.nih.gov/articles/PMC7434642/",
        "source":   "Nurse_Education_Evidence_Review",
        "category": "Research"
    },
]

# =============================================================================
# Download pipeline
# Attempts each source in turn
# Skips files already present and valid (size > 5KB)
# Logs every outcome for full transparency
# =============================================================================
downloaded = []
failed     = []

# Count sources by category
categories = {}
for doc in ALL_SOURCES:
    cat = doc["category"]
    categories[cat] = categories.get(cat, 0) + 1

print(f"Total sources defined: {len(ALL_SOURCES)}")
print(f"By category:")
for cat, count in categories.items():
    print(f"  {cat}: {count}")
print(f"\nSaving to: {os.path.abspath(GUIDELINES_FOLDER)}")
print("-" * 60)

for doc in tqdm(ALL_SOURCES, desc="Downloading"):
    filename = f"{doc['source']}.pdf"
    filepath = os.path.join(GUIDELINES_FOLDER, filename)

    # Skip if already downloaded and valid
    if os.path.exists(filepath) and os.path.getsize(filepath) > 5000:
        downloaded.append(doc)
        continue

    try:
        req = urllib.request.Request(
            doc["url"],
            headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        )
        with urllib.request.urlopen(req, context=ssl_context, timeout=30) as response:
            content = response.read()

        if len(content) > 5000:
            with open(filepath, "wb") as f:
                f.write(content)
            downloaded.append(doc)
        else:
            failed.append(doc)

    except Exception:
        failed.append(doc)

    time.sleep(0.5)

# Summary by category
downloaded_cats = {}
for doc in downloaded:
    cat = doc["category"]
    downloaded_cats[cat] = downloaded_cats.get(cat, 0) + 1

print(f"\nDownload Complete:")
print(f"  Successfully downloaded: {len(downloaded)}")
print(f"  Failed (excluded):       {len(failed)}")
print(f"\nDownloaded by category:")
for cat, count in downloaded_cats.items():
    print(f"  {cat}: {count}")

if failed:
    print(f"\nFailed sources (excluded from corpus):")
    for d in failed:
        print(f"  [{d['category']}] {d['source']}")

Total sources defined: 75
By category:
  NICE: 14
  NHS England: 6
  NWCSP: 2
  Wounds UK: 6
  NHS Trust: 25
  NHS Scotland: 5
  International: 2
  Research: 15

Saving to: C:\Users\MSC1\Desktop\Semester2_Assignments\7146_Assignments\coursework2\cw2_submit_final\knowledge_base
------------------------------------------------------------


Downloading: 100%|█████████████████████████████████████████████████████████████████████| 75/75 [01:03<00:00,  1.18it/s]


Download Complete:
  Successfully downloaded: 74
  Failed (excluded):       1

Downloaded by category:
  NICE: 14
  NHS England: 5
  NWCSP: 2
  Wounds UK: 6
  NHS Trust: 25
  NHS Scotland: 5
  International: 2
  Research: 15

Failed sources (excluded from corpus):
  [NHS England] NHS_Safeguarding_PU


## 1.4 Corpus Statistics

This cell verifies the downloaded knowledge base — counting files by 
category, total size, and confirming the corpus is ready for text 
extraction. A minimum file size check identifies any files that may 
have downloaded as empty or corrupt.

In [8]:
# =============================================================================
# Knowledge Base Verification
# Confirms all downloaded files are valid and reports corpus statistics.
# =============================================================================

files = [f for f in os.listdir(GUIDELINES_FOLDER) if f.endswith('.pdf')]
files.sort()

total_size_mb = sum(
    os.path.getsize(os.path.join(GUIDELINES_FOLDER, f))
    for f in files
) / 1024 / 1024

# Count by category using ALL_SOURCES lookup
source_to_category = {doc['source']: doc['category'] for doc in ALL_SOURCES}
category_counts = {}
for f in files:
    source = f.replace('.pdf', '')
    cat = source_to_category.get(source, 'Unknown')
    category_counts[cat] = category_counts.get(cat, 0) + 1

print(f"Knowledge Base Summary:")
print(f"  Total documents: {len(files)}")
print(f"  Total size:      {total_size_mb:.1f} MB")
print(f"\nBy category:")
for cat, count in sorted(category_counts.items()):
    print(f"  {cat}: {count}")

# Flag any suspiciously small files
print(f"\nFile size check:")
small_files = []
for f in files:
    size_kb = os.path.getsize(os.path.join(GUIDELINES_FOLDER, f)) / 1024
    if size_kb < 10:
        small_files.append((f, size_kb))

if small_files:
    print(f"  WARNING — suspiciously small files detected:")
    for f, size in small_files:
        print(f"    {f}: {size:.1f} KB")
else:
    print(f"  All files passed size check.")

print(f"\nKnowledge base is ready for text extraction.")

Knowledge Base Summary:
  Total documents: 74
  Total size:      40.5 MB

By category:
  International: 2
  NHS England: 5
  NHS Scotland: 5
  NHS Trust: 25
  NICE: 14
  NWCSP: 2
  Research: 15
  Wounds UK: 6

File size check:
  All files passed size check.

Knowledge base is ready for text extraction.


## 1.5 Text Extraction and Preprocessing

Raw text is extracted from all 74 documents in the knowledge base. 
The extraction strategy varies by file type, since the downloaded 
documents include both PDF files and HTML pages saved with a .pdf 
extension.

**PDF extraction** uses pypdf, which reads each page sequentially 
and concatenates the resulting text strings. pypdf is selected over 
alternatives such as pdfminer because it is lightweight, does not 
require system-level dependencies, and produces adequate extraction 
quality for structured clinical documents. A known limitation of 
pypdf is that it extracts text linearly from the PDF object stream, 
which means that spatial relationships between elements — such as 
the colour-coded columns in flowchart documents like the DBTH 
clinical pathway — are lost. The extracted text from such documents 
may therefore appear as a disjointed sequence of fragments rather 
than coherent clinical statements.

To address this limitation, documents identified as flowchart or 
pathway-style PDFs are passed through a restructuring step using 
Claude Haiku. Claude Haiku receives the raw extracted text and 
reconstructs it as a sequence of coherent numbered clinical 
statements, preserving all category numbers, frequency values, 
decision criteria, and threshold values. This restructuring step 
converts spatially-encoded clinical information into linearly-encoded 
prose that chunks and generates QA pairs reliably.

**HTML extraction** uses BeautifulSoup with the html.parser backend. 
BeautifulSoup strips HTML tags, navigation elements, headers, 
footers, and script content, returning the main body text. This 
approach is applied to PubMed Central research articles and web-based 
NHS guidance chapters that were downloaded as HTML content.

**Text cleaning** is applied uniformly after extraction regardless 
of file type. The cleaning function:
- Standardises clinical synonyms using the term map defined in 
  configuration (pressure sore, bed sore, decubitus ulcer all 
  mapped to pressure ulcer)
- Removes Page X of Y page numbering patterns
- Removes isolated single non-alphabetic characters introduced 
  during PDF parsing
- Normalises whitespace
- Preserves all numerical values — repositioning frequencies, 
  category numbers, risk score thresholds, and statistical values 
  are all retained

Documents that produce fewer than 100 words after cleaning are 
logged as extraction failures and excluded from the corpus. This 
threshold identifies documents that are image-only PDFs, 
heavily corrupted extractions, or documents that contained 
predominantly non-textual content.

In [9]:
# =============================================================================
# Text Extraction and Preprocessing
# Extracts text from all 74 documents using pypdf (PDF) or
# BeautifulSoup (HTML). Flowchart documents are restructured via
# Claude Haiku to recover spatial clinical information.
# Numbers are preserved throughout — clinically critical values
# must not be stripped.
# =============================================================================

# Flowchart/pathway documents that require Claude Haiku restructuring
# These are identified by their complex visual layout which pypdf
# cannot recover as coherent clinical statements
FLOWCHART_SOURCES = {
    "DBTH_NHS_2024",
    "DBTH_aSSKINg_Red_2024",
    "DBTH_aSSKINg_Amber_2024",
    "DBTH_aSSKINg_Green_2024",
    "DBTH_PU_Categorisation_2024",
    "DBTH_Product_Selection_2024",
    "DBTH_Foot_Ulcer_2024",
    "NWCSP_Categorisation_2024",
}

def clean_text(text):
    """
    Cleans extracted text while preserving all clinically critical
    numerical values. Only confirmed PDF artefacts are removed.
    """
    # Standardise clinical synonyms
    for synonym, standard in TERM_MAP.items():
        text = re.sub(
            r'\b' + re.escape(synonym) + r'\b',
            standard,
            text,
            flags=re.IGNORECASE
        )

    # Remove Page X of Y patterns only
    text = re.sub(r'[Pp]age\s+\d+\s+of\s+\d+', ' ', text)

    # Remove isolated single non-alphabetic characters
    # (PDF parsing artefacts — does NOT remove numbers)
    text = re.sub(r'(?<!\w)[^a-zA-Z0-9\s](?!\w)', ' ', text)

    # Normalise whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def extract_text_from_pdf(filepath):
    """
    Extracts text from a PDF file using pypdf.
    Returns concatenated text from all pages.
    """
    try:
        reader = pypdf.PdfReader(filepath)
        pages = []
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                pages.append(page_text)
        return '\n'.join(pages)
    except Exception as e:
        return ""


def extract_text_from_html(filepath):
    """
    Extracts text from HTML content saved as .pdf extension.
    Uses BeautifulSoup to strip markup and return body text.
    """
    try:
        with open(filepath, 'rb') as f:
            content = f.read()

        # Detect HTML content
        if b'<html' in content[:2000].lower() or b'<!doctype' in content[:200].lower():
            soup = BeautifulSoup(content, 'html.parser')

            # Remove navigation, scripts, styles
            for tag in soup(['nav', 'script', 'style', 'header',
                             'footer', 'aside', 'menu']):
                tag.decompose()

            return soup.get_text(separator=' ')
        return None

    except Exception:
        return None


def restructure_flowchart(raw_text, source, client):
    """
    Uses Claude Haiku to reconstruct flowchart/pathway documents
    as coherent numbered clinical statements. Preserves all category
    numbers, frequency values, decision criteria, and thresholds.
    """
    try:
        response = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=2000,
            system="""You are a clinical document processor. 
            Convert the following extracted PDF text from a clinical 
            pathway or care plan into clear numbered clinical statements. 
            Preserve ALL: category numbers, frequency values (e.g. every 
            2 hours), threshold scores, decision criteria, care plan names, 
            and clinical actions. Write as clear prose paragraphs. 
            Do not add information not present in the source text.""",
            messages=[{
                "role": "user",
                "content": f"Restructure this clinical pathway text into clear numbered statements:\n\n{raw_text[:3000]}"
            }]
        )
        return response.content[0].text
    except Exception:
        return raw_text


# Initialise Anthropic client for flowchart restructuring
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# =============================================================================
# Main extraction loop
# =============================================================================
corpus = []
extraction_failed = []

files = sorted([f for f in os.listdir(GUIDELINES_FOLDER) if f.endswith('.pdf')])

print(f"Extracting text from {len(files)} documents...")
print("-" * 60)

for filename in tqdm(files, desc="Extracting"):
    filepath  = os.path.join(GUIDELINES_FOLDER, filename)
    source_id = filename.replace('.pdf', '')

    # Try HTML extraction first
    text = extract_text_from_html(filepath)

    # Fall back to PDF extraction
    if not text:
        text = extract_text_from_pdf(filepath)

    # Skip if extraction produced nothing
    if not text or len(text.strip()) < 50:
        extraction_failed.append(source_id)
        continue

    # Restructure flowchart documents via Claude Haiku
    if source_id in FLOWCHART_SOURCES:
        text = restructure_flowchart(text, source_id, client)
        time.sleep(0.5)  # Rate limiting

    # Clean text — preserving numbers
    text = clean_text(text)

    # Skip if cleaned text is too short to produce meaningful chunks
    word_count = len(text.split())
    if word_count < 100:
        extraction_failed.append(source_id)
        continue

    # Get category from ALL_SOURCES lookup
    source_to_meta = {doc['source']: doc for doc in ALL_SOURCES}
    meta = source_to_meta.get(source_id, {})

    corpus.append({
        "source":   source_id,
        "title":    meta.get('title', source_id),
        "category": meta.get('category', 'Unknown'),
        "text":     text,
        "words":    word_count
    })

# Save corpus to disk
with open(CORPUS_FILE, 'w', encoding='utf-8') as f:
    json.dump(corpus, f, indent=2, ensure_ascii=False)

# Summary
total_words = sum(doc['words'] for doc in corpus)
print(f"\nExtraction Complete:")
print(f"  Documents extracted:  {len(corpus)}")
print(f"  Extraction failures:  {len(extraction_failed)}")
print(f"  Total words:          {total_words:,}")
print(f"  Average words/doc:    {total_words // len(corpus):,}")
print(f"  Corpus saved to:      {CORPUS_FILE}")

if extraction_failed:
    print(f"\nFailed extractions (excluded from corpus):")
    for s in extraction_failed:
        print(f"  - {s}")

# Category breakdown
cat_words = {}
for doc in corpus:
    cat = doc['category']
    cat_words[cat] = cat_words.get(cat, 0) + doc['words']

print(f"\nWords by category:")
for cat, words in sorted(cat_words.items()):
    print(f"  {cat}: {words:,} words")

Extracting text from 74 documents...
------------------------------------------------------------


Extracting: 100%|██████████████████████████████████████████████████████████████████████| 74/74 [01:49<00:00,  1.48s/it]


Extraction Complete:
  Documents extracted:  71
  Extraction failures:  3
  Total words:          625,400
  Average words/doc:    8,808
  Corpus saved to:      ./corpus.json

Failed extractions (excluded from corpus):
  - NHS_Wales_PU
  - WWL_NHS_Policy_2025
  - WWL_NHS_SOP_2025

Words by category:
  International: 213,575 words
  NHS England: 16,828 words
  NHS Scotland: 15,734 words
  NHS Trust: 84,529 words
  NICE: 38,958 words
  NWCSP: 3,806 words
  Research: 212,718 words
  Wounds UK: 39,252 words


## 1.6 Document Chunking

The extracted corpus is segmented into overlapping fixed-size chunks 
suitable for QA pair generation and subsequent FAISS indexing. 
Chunking is a critical preprocessing step because BioBERT operates 
on passages of limited length — the model architecture supports a 
maximum of 512 WordPiece tokens, which approximates 380 to 420 words 
of clinical text. Presenting full documents to the QA generation 
model would produce answer spans that are too long or too contextually 
diffuse to serve as reliable training examples.

Chunks are generated using a sliding window approach with a window 
size of 400 words and a stride of 300 words, producing 100-word 
overlaps between consecutive chunks. The overlap serves a specific 
purpose: clinical statements frequently span sentence boundaries 
that coincide with chunk boundaries in a non-overlapping scheme. 
A repositioning instruction such as "patients at high risk must be 
repositioned every 2 hours, with documented rationale for any 
deviation from this schedule" may be split across two non-overlapping 
chunks, with "repositioned every 2 hours" in one chunk and "with 
documented rationale" in the next. The 100-word overlap ensures 
such statements are fully captured in at least one chunk, and 
therefore available as valid answer spans during QA generation.

Chunks with fewer than 50 words after splitting are discarded. 
These typically arise at document boundaries where the final 
window contains only a partial sentence or a footer fragment. 
Retaining such chunks would generate low-quality QA pairs with 
insufficient context for meaningful question formation.

Each chunk retains its source document identifier, title, and 
category for traceability — allowing generated QA pairs to be 
traced back to their originating document.

In [10]:
# =============================================================================
# Document Chunking
# Segments the corpus into overlapping 400-word chunks using a
# sliding window approach. Each chunk retains source metadata
# for traceability. Short chunks are discarded.
# =============================================================================

def chunk_document(doc, chunk_size=CHUNK_SIZE,
                   overlap=CHUNK_OVERLAP,
                   min_words=MIN_CHUNK_WORDS):
    """
    Splits a document into overlapping fixed-size word chunks.
    Returns a list of chunk dictionaries with source metadata.
    """
    words  = doc['text'].split()
    stride = chunk_size - overlap
    chunks = []
    idx    = 0

    while idx < len(words):
        chunk_words = words[idx: idx + chunk_size]

        # Discard chunks that are too short
        if len(chunk_words) >= min_words:
            chunks.append({
                "chunk_id": f"{doc['source']}_{len(chunks)}",
                "source":   doc['source'],
                "title":    doc['title'],
                "category": doc['category'],
                "text":     ' '.join(chunk_words),
                "words":    len(chunk_words)
            })

        idx += stride

    return chunks


# =============================================================================
# Chunk all documents in the corpus
# =============================================================================
all_chunks = []

for doc in tqdm(corpus, desc="Chunking"):
    doc_chunks = chunk_document(doc)
    all_chunks.extend(doc_chunks)

# Save chunks to disk
with open(CHUNKS_FILE, 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

# Summary statistics
total_chunks = len(all_chunks)
avg_words    = sum(c['words'] for c in all_chunks) // total_chunks

# Chunks by category
cat_chunks = {}
for chunk in all_chunks:
    cat = chunk['category']
    cat_chunks[cat] = cat_chunks.get(cat, 0) + 1

print(f"Chunking Complete:")
print(f"  Total chunks produced:  {total_chunks:,}")
print(f"  Average words/chunk:    {avg_words}")
print(f"  Chunks saved to:        {CHUNKS_FILE}")
print(f"\nChunks by category:")
for cat, count in sorted(cat_chunks.items()):
    print(f"  {cat}: {count:,}")

# Projection for QA generation
projected_pairs = total_chunks * QA_PAIRS_PER_CHUNK
print(f"\nQA Generation Projection:")
print(f"  Chunks x {QA_PAIRS_PER_CHUNK} pairs/chunk = {projected_pairs:,} candidate pairs")
print(f"  Target pairs: {TARGET_QA_PAIRS:,}")
print(f"  Estimated chunks needed: {TARGET_QA_PAIRS // QA_PAIRS_PER_CHUNK:,}")

Chunking: 100%|███████████████████████████████████████████████████████████████████████| 71/71 [00:00<00:00, 971.28it/s]

Chunking Complete:
  Total chunks produced:  2,109
  Average words/chunk:    393
  Chunks saved to:        ./chunks.json

Chunks by category:
  International: 712
  NHS England: 57
  NHS Scotland: 55
  NHS Trust: 292
  NICE: 135
  NWCSP: 13
  Research: 713
  Wounds UK: 132

QA Generation Projection:
  Chunks x 5 pairs/chunk = 10,545 candidate pairs
  Target pairs: 6,000
  Estimated chunks needed: 1,200


## 1.7 QA Pair Generation

Question-answer pairs are generated from the full chunked corpus 
using the Claude Haiku large language model via the Anthropic API. 
All 2,109 chunks are processed to maximise dataset size and ensure 
every category is represented proportionally in the final dataset. 
The generation pipeline is designed around three core requirements: 
clinical realism, extractive grounding, and quality filtering.

### Why All Chunks Are Processed

Processing all available chunks rather than stopping at an arbitrary 
pair count ensures that the dataset reflects the full breadth of the 
knowledge base. Stopping early after shuffling risks under-representing 
certain document categories — for example, stopping after 1,200 chunks 
could leave the NWCSP or NHS Scotland categories with very few training 
examples, producing a model that performs poorly on questions drawn 
from those sources. Processing all chunks and accepting the resulting 
dataset size produces a more balanced and representative training set.

### Generation Strategy

Each chunk is submitted to Claude Haiku with a structured prompt 
requesting 3 question-answer pairs. The prompt enforces three 
requirements critical to training data quality.

First, every answer must be an exact verbatim span extracted 
directly from the source passage. The answer string must appear 
character-for-character in the context text. This grounding 
requirement is essential for an extractive QA model — BioBERT 
learns to identify spans within passages, and training examples 
where the answer is paraphrased or inferred rather than directly 
present would introduce a fundamental mismatch between training 
signal and model architecture.

Second, questions must reflect the natural language a healthcare 
practitioner would use at the point of care — asking about clinical 
actions, assessment criteria, risk thresholds, categorisation rules, 
treatment decisions, and repositioning schedules. The prompt 
explicitly discourages questions about document structure, authorship, 
or administrative metadata.

Third, output is required in structured JSON format with a defined 
schema. This eliminates brittle regex-based parsing and allows 
immediate programmatic validation of every pair before retention.

### Quality Filtering

Every generated pair passes through a four-stage validation 
pipeline before being retained:

- **Grounding check**: the answer string must appear verbatim 
  in the context passage — pairs that fail this check are the 
  primary target of filtering, as they indicate Claude Haiku 
  paraphrased rather than extracted
- **Length check**: the answer must contain between 4 and 50 
  words — very short answers lack clinical specificity and 
  very long answers exceed the span length BioBERT handles 
  reliably
- **Format check**: the question must end with a question mark
- **Trivial question check**: the question must be at least 
  10 characters to eliminate degenerate outputs

Pairs failing any check are discarded regardless of apparent 
quality. Based on prototype evaluation, a retention rate of 
60 to 75 percent is anticipated for clinical guideline text, 
substantially higher than the 0.75 percent observed with 
T5-based generation on research abstracts.

### Incremental Saving

Generated pairs are saved to disk every 100 chunks processed. 
This incremental saving strategy prevents total data loss in the 
event of an API error, network interruption, or kernel crash 
mid-pipeline — a practical necessity given the extended runtime 
of this cell.

### API Rate Limiting

A 0.5 second delay is applied between API calls to respect 
Anthropic rate limits. At this rate the full pipeline is 
estimated to complete in approximately 20 to 35 minutes 
depending on API response latency.

In [12]:
# =============================================================================
# QA Pair Generation
# Generates extractive QA pairs from all chunks using Claude Haiku.
# All chunks are processed to ensure full category coverage.
# Claude Haiku is instructed to generate only pairs the passage
# genuinely supports — quality over quantity.
# Each answer is validated as a verbatim span in the source context.
# Pairs are saved incrementally every 100 chunks to prevent data loss.
# =============================================================================

def generate_qa_pairs(chunk, client, n_pairs=QA_PAIRS_PER_CHUNK):
    """
    Submits a single chunk to Claude Haiku and requests up to n_pairs
    QA pairs. Claude Haiku is instructed to generate fewer pairs if
    the passage does not genuinely support the full quota.
    Validates each pair through grounding, length, and format checks.
    Returns a list of validated QA dictionaries.
    """
    prompt = f"""You are a clinical QA dataset generator for a pressure ulcer 
training system used by NHS nurses and healthcare practitioners.

Given the following clinical text passage, generate up to {n_pairs} 
question-answer pairs. Generate fewer than {n_pairs} pairs if the passage 
does not genuinely support that many meaningful clinical questions. 
Do not pad to fill a quota — quality is more important than quantity.

STRICT REQUIREMENTS:
1. Only generate a pair if the passage contains a clear, specific clinical 
   fact that a nurse or healthcare practitioner would genuinely ask about — 
   such as a risk threshold, repositioning frequency, wound category 
   definition, dressing type, assessment tool, or care action.
2. Every answer must be copied VERBATIM and EXACTLY from the passage below.
   Do not paraphrase, summarise, or infer. Copy the exact words.
3. Answers must be between {MIN_ANSWER_WORDS} and {MAX_ANSWER_WORDS} words.
4. Do not generate pairs about document authorship, review dates, trust 
   names, policy numbers, or administrative content.
5. Return ONLY valid JSON. No preamble, no explanation, no markdown fences.

Return this exact JSON format:
{{
  "pairs": [
    {{"question": "...", "answer": "..."}},
    {{"question": "...", "answer": "..."}}
  ]
}}

PASSAGE:
{chunk['text']}"""

    try:
        response = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=1500,
            messages=[{"role": "user", "content": prompt}]
        )

        raw = response.content[0].text.strip()

        # Strip markdown code fences if present
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'^```\s*',     '', raw)
        raw = re.sub(r'\s*```$',     '', raw)

        data  = json.loads(raw)
        pairs = data.get('pairs', [])

        validated = []
        for pair in pairs:
            q = pair.get('question', '').strip()
            a = pair.get('answer',   '').strip()

            # Grounding check — answer must appear verbatim in context
            if a not in chunk['text']:
                continue

            # Length check — answer must be within word count bounds
            a_words = len(a.split())
            if a_words < MIN_ANSWER_WORDS or a_words > MAX_ANSWER_WORDS:
                continue

            # Format check — question must end with question mark
            if not q.endswith('?'):
                continue

            # Trivial question check — question must be meaningful length
            if len(q) < 10:
                continue

            # Duplicate check — discard if same question already exists
            existing_questions = {p['question'].lower() for p in validated}
            if q.lower() in existing_questions:
                continue

            # Find answer start position for SQuAD format
            answer_start = chunk['text'].find(a)

            validated.append({
                "question":     q,
                "answer":       a,
                "context":      chunk['text'],
                "answer_start": answer_start,
                "source":       chunk['source'],
                "title":        chunk['title'],
                "category":     chunk['category'],
                "chunk_id":     chunk['chunk_id']
            })

        return validated

    except Exception:
        return []


# =============================================================================
# Main generation loop
# Processes all chunks — no early stopping
# Ensures full category coverage across the dataset
# =============================================================================
qa_pairs         = []
chunks_processed = 0
chunks_skipped   = 0

# Shuffle chunks to distribute categories across the generation run
# This ensures incremental saves contain balanced category coverage
random.shuffle(all_chunks)

print(f"Starting QA generation...")
print(f"Chunks to process:     {len(all_chunks):,}")
print(f"Max pairs per chunk:   {QA_PAIRS_PER_CHUNK}")
print(f"Max candidate pairs:   {len(all_chunks) * QA_PAIRS_PER_CHUNK:,}")
print("-" * 60)

for i, chunk in enumerate(tqdm(all_chunks, desc="Generating QA pairs")):

    pairs = generate_qa_pairs(chunk, client)

    if pairs:
        qa_pairs.extend(pairs)
        chunks_processed += 1
    else:
        chunks_skipped += 1

    # Save incrementally every 100 chunks to prevent data loss
    if (i + 1) % 100 == 0:
        with open(QA_PAIRS_FILE, 'w', encoding='utf-8') as f:
            json.dump(qa_pairs, f, indent=2, ensure_ascii=False)
        tqdm.write(f"  Checkpoint [{i+1}/{len(all_chunks)}]: "
                   f"{len(qa_pairs):,} pairs retained so far")

    time.sleep(0.5)

# Final save
with open(QA_PAIRS_FILE, 'w', encoding='utf-8') as f:
    json.dump(qa_pairs, f, indent=2, ensure_ascii=False)

# Calculate retention rate
total_candidates = chunks_processed * QA_PAIRS_PER_CHUNK
retention_rate   = (len(qa_pairs) / total_candidates * 100) if total_candidates > 0 else 0

# Category breakdown
cat_pairs = {}
for pair in qa_pairs:
    cat = pair['category']
    cat_pairs[cat] = cat_pairs.get(cat, 0) + 1

print(f"\nQA Generation Complete:")
print(f"  Total chunks:              {len(all_chunks):,}")
print(f"  Chunks processed:          {chunks_processed:,}")
print(f"  Chunks skipped (no pairs): {chunks_skipped:,}")
print(f"  Answerable pairs retained: {len(qa_pairs):,}")
print(f"  Retention rate:            {retention_rate:.1f}%")
print(f"  QA pairs saved to:         {QA_PAIRS_FILE}")
print(f"\nPairs by category:")
for cat, count in sorted(cat_pairs.items()):
    print(f"  {cat}: {count:,}")

Starting QA generation...
Chunks to process:     2,109
Max pairs per chunk:   3
Max candidate pairs:   6,327
------------------------------------------------------------


Generating QA pairs:   5%|██▊                                                        | 99/2109 [03:31<50:24,  1.50s/it]

  Checkpoint [100/2109]: 99 pairs retained so far


Generating QA pairs:   9%|█████▎                                                  | 199/2109 [06:56<1:03:58,  2.01s/it]

  Checkpoint [200/2109]: 194 pairs retained so far


...

  Checkpoint [2000/2109]: 1,923 pairs retained so far


Generating QA pairs: 100%|██████████████████████████████████████████████████████▋| 2099/2109 [1:14:09<00:20,  2.08s/it]

  Checkpoint [2100/2109]: 2,043 pairs retained so far


Generating QA pairs: 100%|███████████████████████████████████████████████████████| 2109/2109 [1:14:29<00:00,  2.12s/it]


QA Generation Complete:
  Total chunks:              2,109
  Chunks processed:          1,163
  Chunks skipped (no pairs): 946
  Answerable pairs retained: 2,055
  Retention rate:            58.9%
  QA pairs saved to:         ./qa_pairs.json

Pairs by category:
  International: 732
  NHS England: 81
  NHS Scotland: 69
  NHS Trust: 373
  NICE: 212
  NWCSP: 11
  Research: 452
  Wounds UK: 125


## 1.7.1 QA Generation — Second Pass

A second pass is performed on chunks that produced valid pairs in 
the first pass. Claude Haiku is instructed to generate different 
questions from those already asked — specifically avoiding any 
question whose answer was already extracted from that chunk. This 
approach maximises the diversity of the training dataset without 
re-processing chunks that produced no pairs in the first pass, 
which were likely too short, too administrative, or too visually 
encoded to yield meaningful clinical QA pairs.

Generation stops automatically when the combined total of first 
and second pass pairs reaches 3,500 — a target that comfortably 
satisfies the assessment specification minimum while remaining 
within the available API credit budget.

In [13]:
# =============================================================================
# QA Generation — Second Pass
# Re-processes chunks that produced valid pairs in the first pass.
# Claude Haiku is instructed to generate NEW questions different
# from those already asked on each chunk.
# Stops when combined total reaches 3,500 pairs.
# =============================================================================

SECOND_PASS_TARGET = 3500  # Stop when total pairs reach this

def generate_qa_pairs_second_pass(chunk, existing_questions, client,
                                   n_pairs=QA_PAIRS_PER_CHUNK):
    """
    Generates new QA pairs from a chunk already processed in pass 1.
    Receives the list of questions already asked so Claude Haiku
    can generate genuinely different questions.
    """
    existing_q_str = "\n".join(f"- {q}" for q in existing_questions)

    prompt = f"""You are a clinical QA dataset generator for a pressure ulcer 
training system used by NHS nurses and healthcare practitioners.

Given the following clinical text passage, generate up to {n_pairs} 
NEW question-answer pairs that are DIFFERENT from the questions already 
asked below.

QUESTIONS ALREADY ASKED — DO NOT REPEAT THESE OR ASK SIMILAR QUESTIONS:
{existing_q_str}

STRICT REQUIREMENTS:
1. Generate questions about different clinical facts from those already 
   covered above. Focus on facts not yet asked about in this passage.
2. Every answer must be copied VERBATIM and EXACTLY from the passage below.
   Do not paraphrase, summarise, or infer. Copy the exact words.
3. Answers must be between {MIN_ANSWER_WORDS} and {MAX_ANSWER_WORDS} words.
4. Do not generate pairs about document authorship, review dates, trust 
   names, policy numbers, or administrative content.
5. If the passage has no remaining facts worth asking about, return an 
   empty pairs list.
6. Return ONLY valid JSON. No preamble, no explanation, no markdown fences.

Return this exact JSON format:
{{
  "pairs": [
    {{"question": "...", "answer": "..."}},
    {{"question": "...", "answer": "..."}}
  ]
}}

PASSAGE:
{chunk['text']}"""

    try:
        response = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=1500,
            messages=[{"role": "user", "content": prompt}]
        )

        raw = response.content[0].text.strip()
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'^```\s*',     '', raw)
        raw = re.sub(r'\s*```$',     '', raw)

        data  = json.loads(raw)
        pairs = data.get('pairs', [])

        validated = []
        for pair in pairs:
            q = pair.get('question', '').strip()
            a = pair.get('answer',   '').strip()

            # Grounding check
            if a not in chunk['text']:
                continue

            # Length check
            a_words = len(a.split())
            if a_words < MIN_ANSWER_WORDS or a_words > MAX_ANSWER_WORDS:
                continue

            # Format check
            if not q.endswith('?'):
                continue

            # Trivial question check
            if len(q) < 10:
                continue

            # Duplicate check against existing questions
            if q.lower() in {eq.lower() for eq in existing_questions}:
                continue

            answer_start = chunk['text'].find(a)

            validated.append({
                "question":     q,
                "answer":       a,
                "context":      chunk['text'],
                "answer_start": answer_start,
                "source":       chunk['source'],
                "title":        chunk['title'],
                "category":     chunk['category'],
                "chunk_id":     chunk['chunk_id']
            })

        return validated

    except Exception:
        return []


# =============================================================================
# Build lookup of chunks that produced pairs in pass 1
# and the questions already asked per chunk
# =============================================================================

# Load existing pairs
with open(QA_PAIRS_FILE, 'r', encoding='utf-8') as f:
    qa_pairs = json.load(f)

print(f"Pairs from first pass: {len(qa_pairs):,}")

# Build dict: chunk_id -> list of questions already asked
chunk_questions = {}
for pair in qa_pairs:
    cid = pair['chunk_id']
    if cid not in chunk_questions:
        chunk_questions[cid] = []
    chunk_questions[cid].append(pair['question'])

# Get chunks that produced valid pairs in pass 1
productive_chunk_ids = set(chunk_questions.keys())
productive_chunks    = [c for c in all_chunks if c['chunk_id'] in productive_chunk_ids]

# Shuffle for category diversity
random.shuffle(productive_chunks)

print(f"Productive chunks available for second pass: {len(productive_chunks):,}")
print(f"Target total pairs: {SECOND_PASS_TARGET:,}")
print(f"New pairs needed:   {SECOND_PASS_TARGET - len(qa_pairs):,}")
print("-" * 60)

# =============================================================================
# Second pass generation loop
# =============================================================================
new_pairs        = []
chunks_processed = 0
chunks_skipped   = 0

for i, chunk in enumerate(tqdm(productive_chunks, desc="Second pass")):

    # Stop when target reached
    if len(qa_pairs) + len(new_pairs) >= SECOND_PASS_TARGET:
        break

    existing_qs = chunk_questions.get(chunk['chunk_id'], [])
    pairs       = generate_qa_pairs_second_pass(chunk, existing_qs, client)

    if pairs:
        new_pairs.extend(pairs)
        chunks_processed += 1
    else:
        chunks_skipped += 1

    # Save incrementally every 100 chunks
    if (i + 1) % 100 == 0:
        combined = qa_pairs + new_pairs
        with open(QA_PAIRS_FILE, 'w', encoding='utf-8') as f:
            json.dump(combined, f, indent=2, ensure_ascii=False)
        tqdm.write(f"  Checkpoint [{i+1}]: "
                   f"{len(qa_pairs) + len(new_pairs):,} total pairs so far")

    time.sleep(0.5)

# Final combined save
all_qa_pairs = qa_pairs + new_pairs
with open(QA_PAIRS_FILE, 'w', encoding='utf-8') as f:
    json.dump(all_qa_pairs, f, indent=2, ensure_ascii=False)

# Summary
cat_pairs = {}
for pair in all_qa_pairs:
    cat = pair['category']
    cat_pairs[cat] = cat_pairs.get(cat, 0) + 1

print(f"\nSecond Pass Complete:")
print(f"  First pass pairs:          {len(qa_pairs):,}")
print(f"  Second pass pairs added:   {len(new_pairs):,}")
print(f"  Total answerable pairs:    {len(all_qa_pairs):,}")
print(f"  Chunks processed:          {chunks_processed:,}")
print(f"  Chunks skipped:            {chunks_skipped:,}")
print(f"  QA pairs saved to:         {QA_PAIRS_FILE}")
print(f"\nPairs by category:")
for cat, count in sorted(cat_pairs.items()):
    print(f"  {cat}: {count:,}")

Pairs from first pass: 2,055
Productive chunks available for second pass: 1,163
Target total pairs: 3,500
New pairs needed:   1,445
------------------------------------------------------------


Second pass:   9%|█████▋                                                             | 99/1163 [04:05<43:32,  2.46s/it]

  Checkpoint [100]: 2,194 total pairs so far


Second pass:  17%|███████████▎                                                      | 199/1163 [07:58<38:57,  2.43s/it]

  Checkpoint [200]: 2,303 total pairs so far


...

  Checkpoint [1000]: 3,369 total pairs so far


Second pass:  94%|█████████████████████████████████████████████████████████████▍   | 1099/1163 [47:00<03:27,  3.24s/it]

  Checkpoint [1100]: 3,495 total pairs so far


Second pass:  95%|█████████████████████████████████████████████████████████████▊   | 1107/1163 [47:17<02:23,  2.56s/it]



Second Pass Complete:
  First pass pairs:          2,055
  Second pass pairs added:   1,445
  Total answerable pairs:    3,500
  Chunks processed:          829
  Chunks skipped:            278
  QA pairs saved to:         ./qa_pairs.json

Pairs by category:
  International: 1,212
  NHS England: 150
  NHS Scotland: 123
  NHS Trust: 639
  NICE: 354
  NWCSP: 23
  Research: 804
  Wounds UK: 195


## 1.8 Unanswerable Question Generation

SQuAD 2.0 requires unanswerable questions alongside answerable ones. 
An unanswerable pair consists of a genuine clinical question paired 
with a context passage from which its answer cannot be extracted. 
Training on unanswerable pairs teaches BioBERT to return a structured 
no-answer response when the retrieved context does not contain the 
relevant information, rather than always returning a span regardless 
of confidence.

Unanswerable pairs are generated using the wrong-context pairing 
method — the standard approach used in the original SQuAD 2.0 
benchmark. For each unanswerable pair, a question is taken from 
one source document and paired with a context passage from a 
different source document. The question is genuinely clinical and 
answerable in its original context, but the paired context does not 
contain the answer. This creates a realistic unanswerable scenario 
that reflects the type of retrieval failure the deployed system 
may encounter — where the FAISS retriever returns a relevant but 
insufficient passage.

The unanswerable ratio is set at 20% of the answerable pairs, 
consistent with the SQuAD 2.0 benchmark proportion. At 3,500 
answerable pairs this produces 700 unanswerable pairs, giving 
a total dataset of 4,200 examples.

A source mismatch check ensures that the context passage used 
for each unanswerable pair genuinely does not contain the answer 
string, preventing accidental valid pairs from entering the 
unanswerable set and corrupting the training signal.

In [14]:
# =============================================================================
# Unanswerable Question Generation
# Creates unanswerable pairs using wrong-context pairing.
# Each question is paired with a context from a different source
# document that does not contain the answer.
# Ratio: 20% of answerable pairs = 700 unanswerable pairs.
# =============================================================================

# Load answerable pairs
with open(QA_PAIRS_FILE, 'r', encoding='utf-8') as f:
    answerable_pairs = json.load(f)

n_unanswerable = int(len(answerable_pairs) * UNANSWERABLES_RATIO)
print(f"Answerable pairs:    {len(answerable_pairs):,}")
print(f"Unanswerable target: {n_unanswerable:,}")
print(f"Total dataset size:  {len(answerable_pairs) + n_unanswerable:,}")
print("-" * 60)

# Build pool of contexts indexed by source
# Used to find a context from a different source for each question
source_contexts = {}
for pair in answerable_pairs:
    src = pair['source']
    if src not in source_contexts:
        source_contexts[src] = []
    source_contexts[src].append(pair['context'])

all_sources = list(source_contexts.keys())

# =============================================================================
# Generate unanswerable pairs
# =============================================================================
unanswerable_pairs = []
attempts           = 0
max_attempts       = n_unanswerable * 10  # Prevent infinite loop

# Sample answerable pairs to use as question sources
sampled = random.sample(answerable_pairs, min(n_unanswerable * 3,
                                               len(answerable_pairs)))

for pair in sampled:
    if len(unanswerable_pairs) >= n_unanswerable:
        break

    attempts += 1
    if attempts > max_attempts:
        break

    question    = pair['question']
    answer      = pair['answer']
    orig_source = pair['source']

    # Find a context from a different source
    other_sources = [s for s in all_sources if s != orig_source]
    if not other_sources:
        continue

    # Try up to 5 different source documents
    for _ in range(5):
        wrong_source  = random.choice(other_sources)
        wrong_context = random.choice(source_contexts[wrong_source])

        # Verify answer is genuinely not present in wrong context
        if answer.lower() not in wrong_context.lower():
            unanswerable_pairs.append({
                "question":      question,
                "answer":        "",
                "context":       wrong_context,
                "answer_start":  -1,
                "is_impossible": True,
                "source":        wrong_source,
                "orig_source":   orig_source,
                "category":      pair['category'],
                "chunk_id":      pair['chunk_id']
            })
            break

# =============================================================================
# Combine and save full dataset
# =============================================================================

# Tag answerable pairs
for pair in answerable_pairs:
    pair['is_impossible'] = False

all_pairs = answerable_pairs + unanswerable_pairs
random.shuffle(all_pairs)

with open(ALL_QA_FILE, 'w', encoding='utf-8') as f:
    json.dump(all_pairs, f, indent=2, ensure_ascii=False)

print(f"Unanswerable pairs generated: {len(unanswerable_pairs):,}")
print(f"Total pairs:                  {len(all_pairs):,}")
print(f"  Answerable:                 {sum(1 for p in all_pairs if not p['is_impossible']):,}")
print(f"  Unanswerable:               {sum(1 for p in all_pairs if p['is_impossible']):,}")
print(f"Full dataset saved to:        {ALL_QA_FILE}")

Answerable pairs:    3,500
Unanswerable target: 700
Total dataset size:  4,200
------------------------------------------------------------
Unanswerable pairs generated: 700
Total pairs:                  4,200
  Answerable:                 3,500
  Unanswerable:               700
Full dataset saved to:        ./all_qa_pairs.json


## 1.9 Deduplication

Before formatting and splitting, the combined dataset is deduplicated 
to remove exact and near-duplicate QA pairs. Duplicates arise from 
two sources in this pipeline.

First, the two-pass generation strategy may produce identical or 
highly similar questions from the same chunk — particularly for 
chunks with limited clinical content where Claude Haiku generates 
the same clinical fact framed as a slightly different question 
across passes.

Second, the knowledge base contains several documents that 
reproduce identical content — for example, the NWCSP clinical 
pathway is hosted by multiple NHS organisations and downloaded 
as separate files. Chunks from these documents will produce 
identical or near-identical context passages, and therefore 
identical QA pairs.

Two levels of deduplication are applied:

**Exact deduplication** removes pairs where both the question 
and answer strings are character-for-character identical. These 
represent genuine duplicates with no added value to the training 
dataset.

**Near-duplicate question deduplication** removes pairs where 
the question string is identical to an existing question 
regardless of context. A question that has already been asked 
and answered adds no new training signal even if the context 
passage is marginally different.

Deduplication is performed before splitting to ensure that 
no duplicate pairs appear across the train, validation, and 
test sets, which would cause artificial inflation of evaluation 
metrics.

In [17]:
# =============================================================================
# Deduplication
# Removes exact duplicate pairs and near-duplicate questions
# before SQuAD formatting and splitting.
# Performed before splitting to prevent cross-split duplicates.
# =============================================================================

# Load combined dataset
with open(ALL_QA_FILE, 'r', encoding='utf-8') as f:
    all_pairs = json.load(f)

print(f"Pairs before deduplication: {len(all_pairs):,}")

# -----------------------------------------------------------------------------
# Level 1: Exact deduplication
# Remove pairs where question AND answer are both identical
# -----------------------------------------------------------------------------
seen_exact  = set()
deduped_l1  = []

for pair in all_pairs:
    key = (pair['question'].strip().lower(), pair['answer'].strip().lower())
    if key not in seen_exact:
        seen_exact.add(key)
        deduped_l1.append(pair)

removed_exact = len(all_pairs) - len(deduped_l1)
print(f"Exact duplicates removed:   {removed_exact:,}")

# -----------------------------------------------------------------------------
# Level 2: Near-duplicate question deduplication
# Remove pairs where the question is identical to an existing one
# regardless of context
# -----------------------------------------------------------------------------
seen_questions = set()
deduped_l2     = []

for pair in deduped_l1:
    q_key = pair['question'].strip().lower()
    if q_key not in seen_questions:
        seen_questions.add(q_key)
        deduped_l2.append(pair)

removed_near = len(deduped_l1) - len(deduped_l2)
print(f"Near-duplicate questions removed: {removed_near:,}")

# Final deduplicated dataset
all_pairs_deduped = deduped_l2

# Recount answerable and unanswerable
n_answerable   = sum(1 for p in all_pairs_deduped if not p['is_impossible'])
n_unanswerable = sum(1 for p in all_pairs_deduped if p['is_impossible'])

print(f"\nAfter deduplication:")
print(f"  Total pairs:         {len(all_pairs_deduped):,}")
print(f"  Answerable:          {n_answerable:,}")
print(f"  Unanswerable:        {n_unanswerable:,}")
print(f"  Total removed:       {len(all_pairs) - len(all_pairs_deduped):,}")

# Save deduplicated dataset back to ALL_QA_FILE
with open(ALL_QA_FILE, 'w', encoding='utf-8') as f:
    json.dump(all_pairs_deduped, f, indent=2, ensure_ascii=False)

print(f"\nDeduplicated dataset saved to: {ALL_QA_FILE}")

Pairs before deduplication: 4,200
Exact duplicates removed:   90
Near-duplicate questions removed: 731

After deduplication:
  Total pairs:         3,379
  Answerable:          3,034
  Unanswerable:        345
  Total removed:       821

Deduplicated dataset saved to: ./all_qa_pairs.json


## 1.10 SQuAD 2.0 Formatting and Dataset Splitting

The combined dataset of 4,200 pairs is converted into SQuAD 2.0 
JSON format and split into training, validation, and test sets 
for use in Task 3 fine-tuning and Task 4 evaluation.

### SQuAD 2.0 Format

SQuAD 2.0 is the standard benchmark format for extractive QA 
and is natively supported by the HuggingFace Transformers library 
used for BioBERT fine-tuning in Task 3. The format organises data 
hierarchically — a top-level data array contains topic groups, 
each topic group contains paragraphs, and each paragraph contains 
a context passage and a list of QA pairs. Unanswerable pairs are 
represented with an empty answers list and an is_impossible flag 
set to True. This structure allows the HuggingFace Trainer to 
handle both answerable and unanswerable examples natively without 
requiring a custom data loader.

### Dataset Splitting

The dataset is split into train, validation, and test sets at a 
70/15/15 ratio — a standard split proportion for supervised 
learning tasks of this scale. The split is performed at the 
pair level after shuffling the full dataset with the fixed random 
seed of 42. This approach guarantees the correct proportions 
regardless of the size distribution of source documents, which 
varies considerably across the corpus — large documents such as 
the NPUAP EPUAP clinical practice guideline and the WHS treatment 
guidelines contribute several hundred pairs each, while shorter 
documents such as individual NICE QS89 chapters contribute 
fewer than ten pairs. A source-level split would assign 
disproportionate pair counts to whichever split received the 
large documents, producing ratios as skewed as 77/7/16 rather 
than the intended 70/15/15.

A limitation of pair-level splitting is that pairs derived from 
the same context passage may appear in both the training and 
evaluation splits, introducing a small degree of context overlap 
between splits. In practice this risk is modest — the 2,109 
chunks in the corpus are distributed across 71 source documents, 
meaning the probability of identical context passages appearing 
in multiple splits is low. Nonetheless this represents a 
methodological limitation that a production system would address 
through stricter context-level deduplication before splitting.

The final SQuAD 2.0 dataset is saved as a single JSON file 
containing train, validation, and test keys, ready for direct 
loading in Task 2.

In [18]:
# =============================================================================
# SQuAD 2.0 Formatting and Dataset Splitting
# Converts all QA pairs into SQuAD 2.0 hierarchical format.
# Splits at pair level after shuffling to ensure correct 70/15/15
# proportions regardless of source document size variation.
# Fixed seed 42 ensures reproducibility across runs.
# =============================================================================

def format_squad(pairs):
    """
    Converts a flat list of QA pair dicts into SQuAD 2.0 format.
    Groups pairs by source document then by context passage to
    produce the required hierarchical structure.
    """
    # Group pairs: source -> context -> list of pairs
    source_groups = {}
    for pair in pairs:
        src = pair['source']
        if src not in source_groups:
            source_groups[src] = {}
        ctx = pair['context']
        if ctx not in source_groups[src]:
            source_groups[src][ctx] = []
        source_groups[src][ctx].append(pair)

    squad_data = []
    for source, contexts in source_groups.items():
        paragraphs = []
        for context, context_pairs in contexts.items():
            qas = []
            for pair in context_pairs:
                if pair['is_impossible']:
                    # Unanswerable pair — empty answers list
                    qas.append({
                        "id":            f"{pair['chunk_id']}_{len(qas)}",
                        "question":      pair['question'],
                        "answers":       [],
                        "is_impossible": True
                    })
                else:
                    # Answerable pair — verbatim span with start position
                    qas.append({
                        "id":            f"{pair['chunk_id']}_{len(qas)}",
                        "question":      pair['question'],
                        "answers": [{
                            "text":         pair['answer'],
                            "answer_start": pair['answer_start']
                        }],
                        "is_impossible": False
                    })

            paragraphs.append({
                "context": context,
                "qas":     qas
            })

        squad_data.append({
            "title":      source,
            "paragraphs": paragraphs
        })

    return squad_data


# =============================================================================
# Pair-level split at 70/15/15
# Shuffle all pairs with fixed seed then slice at correct boundaries
# =============================================================================

# Load full dataset
with open(ALL_QA_FILE, 'r', encoding='utf-8') as f:
    all_pairs = json.load(f)

# Shuffle at pair level with fixed seed for reproducibility
random.seed(SEED)
random.shuffle(all_pairs)

total   = len(all_pairs)
n_train = int(total * TRAIN_RATIO)
n_val   = int(total * VAL_RATIO)

train_pairs = all_pairs[:n_train]
val_pairs   = all_pairs[n_train : n_train + n_val]
test_pairs  = all_pairs[n_train + n_val:]

print(f"Dataset split:")
print(f"  Total pairs: {total:,}")
print(f"  Train:       {len(train_pairs):,} ({len(train_pairs)/total*100:.1f}%)")
print(f"  Validation:  {len(val_pairs):,}   ({len(val_pairs)/total*100:.1f}%)")
print(f"  Test:        {len(test_pairs):,}   ({len(test_pairs)/total*100:.1f}%)")

# Format each split into SQuAD 2.0 structure
squad_dataset = {
    "train":      {"version": "v2.0", "data": format_squad(train_pairs)},
    "validation": {"version": "v2.0", "data": format_squad(val_pairs)},
    "test":       {"version": "v2.0", "data": format_squad(test_pairs)}
}

# Save to disk
with open(SQUAD_FILE, 'w', encoding='utf-8') as f:
    json.dump(squad_dataset, f, indent=2, ensure_ascii=False)

# =============================================================================
# Verification — count pairs per split from the saved SQuAD structure
# =============================================================================
def count_squad_pairs(squad_split):
    total        = 0
    answerable   = 0
    unanswerable = 0
    for article in squad_split['data']:
        for para in article['paragraphs']:
            for qa in para['qas']:
                total += 1
                if qa['is_impossible']:
                    unanswerable += 1
                else:
                    answerable += 1
    return total, answerable, unanswerable

tr_total, tr_ans, tr_unans = count_squad_pairs(squad_dataset['train'])
vl_total, vl_ans, vl_unans = count_squad_pairs(squad_dataset['validation'])
te_total, te_ans, te_unans = count_squad_pairs(squad_dataset['test'])

grand_total = tr_total + vl_total + te_total

print(f"\nSQuAD 2.0 Dataset Verification:")
print(f"{'Split':<15} {'Total':>8} {'Answerable':>12} "
      f"{'Unanswerable':>14} {'Ratio':>8}")
print(f"{'-'*60}")
print(f"{'Train':<15} {tr_total:>8,} {tr_ans:>12,} "
      f"{tr_unans:>14,} {tr_total/grand_total*100:>7.1f}%")
print(f"{'Validation':<15} {vl_total:>8,} {vl_ans:>12,} "
      f"{vl_unans:>14,} {vl_total/grand_total*100:>7.1f}%")
print(f"{'Test':<15} {te_total:>8,} {te_ans:>12,} "
      f"{te_unans:>14,} {te_total/grand_total*100:>7.1f}%")
print(f"{'-'*60}")
print(f"{'Total':<15} {grand_total:>8,} "
      f"{tr_ans+vl_ans+te_ans:>12,} "
      f"{tr_unans+vl_unans+te_unans:>14,}")
print(f"\nDataset saved to: {SQUAD_FILE}")

Dataset split:
  Total pairs: 3,379
  Train:       2,365 (70.0%)
  Validation:  506   (15.0%)
  Test:        508   (15.0%)

SQuAD 2.0 Dataset Verification:
Split              Total   Answerable   Unanswerable    Ratio
------------------------------------------------------------
Train              2,365        2,133            232    70.0%
Validation           506          445             61    15.0%
Test                 508          456             52    15.0%
------------------------------------------------------------
Total              3,379        3,034            345

Dataset saved to: ./squad_dataset.json


## 1.11 Task 1 Summary and Critical Discussion

### Dataset Summary

The dataset construction pipeline successfully produced a SQuAD 2.0 
formatted dataset from 71 authoritative clinical documents covering 
pressure ulcer prevention, assessment, classification, and management. 
The key statistics are summarised below:

| Metric | Value |
|--------|-------|
| Source documents downloaded | 74 |
| Documents successfully extracted | 71 |
| Total words in corpus | 625,400 |
| Chunks produced | 2,109 |
| Answerable pairs before deduplication | 3,500 |
| Unanswerable pairs before deduplication | 700 |
| Exact duplicates removed | — |
| Near-duplicate questions removed | — |
| Total pairs after deduplication | — |
| Training pairs | — (70%) |
| Validation pairs | — (15%) |
| Test pairs | — (15%) |

*Final counts populated programmatically in Cell 23.*

### What Worked Well

The shift from research abstracts to clinical guidelines as the 
primary knowledge source produced a substantially more clinically 
relevant corpus than the initial prototype. The direct imperative 
language of NHS Trust tissue viability policies, NICE quality 
standards, and NWCSP pathway documents generated questions that 
reflect genuine nursing practice — questions about repositioning 
frequency, wound categorisation, risk score thresholds, and care 
plan actions rather than the statistical methodology questions 
produced from research abstracts in the prototype.

The verbatim grounding requirement proved effective — a retention 
rate of 58.9% on processed chunks indicates that Claude Haiku 
was sufficiently constrained to produce extractive rather than 
generative answers in the majority of cases, directly addressing 
the context mismatch failure observed with T5 in the prototype.

The flowchart restructuring step using Claude Haiku recovered 
clinically important information from pathway documents such as 
the DBTH clinical pathway and aSSKINg care plans that would 
otherwise have been lost to spatially-encoded PDF extraction.

Deduplication at both the exact pair level and the question 
level removed redundant training examples that would otherwise 
have inflated apparent dataset size without contributing 
additional training signal. This step is particularly important 
given the two-pass generation strategy, which increases the 
probability of semantically similar questions being generated 
from the same chunk across passes.

### Limitations and Critical Discussion

Several limitations of the dataset construction approach warrant 
acknowledgement.

**Dataset size.** The final dataset, while within the assessment specification range of 3,000 to 8,000 pairs, is at the lower end of what would be considered adequate for fine-tuning a large transformer model in a production context. BioBERT Large has 340 million parameters and benefits from substantially larger fine-tuning datasets — the original BioBERT paper fine-tuned on QA datasets of 87,000 to 100,000 examples. The dataset size in this implementation reflects a deliberate trade-off between generation quality and volume. The two-pass generation strategy with verbatim grounding validation and quality filtering retained approximately 58% of candidate pairs, prioritising clinical precision over scale. A higher volume dataset could be produced from the same corpus by relaxing the quality filtering thresholds or increasing pairs per chunk, but doing so would introduce clinically imprecise or weakly grounded training examples — an unacceptable trade-off in a patient safety context where answer accuracy is more critical than dataset scale. The conservative filtering approach is therefore considered the appropriate design choice for this clinical application, with the acknowledged limitation that model generalisation would benefit from a larger corpus of source documents to increase the natural ceiling on high-quality pair production.

**Chunk skipping rate.** A total of 946 chunks (44.9%) produced 
no QA pairs in the first generation pass. This high skip rate 
reflects the conservative prompt instruction to generate fewer 
pairs when the passage does not genuinely support them. While 
this improved individual pair quality, it reduced overall dataset 
coverage. This means that clinical content present in those 
946 chunks — potentially including important information on 
specific wound care interventions or equipment selection — is 
absent from the training dataset.

**Category imbalance.** The dataset contains a notable imbalance 
across source categories — International guidelines contribute 
a disproportionately large share of pairs relative to NWCSP 
documents which contribute fewer than 25 pairs. This reflects 
the size disparity between source documents rather than a 
deliberate weighting decision. This imbalance may cause the 
trained model to perform better on questions reflecting 
international guideline language than on questions derived 
from the current NHS clinical pathway.

**Wrong-context unanswerable generation.** The wrong-context 
pairing method produces unanswerable examples that are somewhat 
artificial. In practice, unanswerable scenarios arise from more 
complex retrieval failures than simple context mismatch. A more 
sophisticated approach would generate genuinely ambiguous 
passages where the answer is related but absent, more closely 
reflecting real deployment scenarios.

**Document currency.** The knowledge base includes documents 
ranging from 2014 to 2025. Older documents such as the NPUAP 
EPUAP 2014 guideline may contain recommendations that have 
since been superseded by more recent NHS guidance. The model 
trained on this corpus may therefore reflect a mixture of 
current and outdated clinical recommendations, which represents 
a patient safety consideration in any real deployment scenario.

**Deduplication completeness.** The deduplication strategy 
applied removes exact duplicates and identical question strings 
but does not address semantic near-duplicates — questions that 
are phrased differently but ask about the same clinical fact. 
Semantic deduplication using embedding-based similarity would 
provide more thorough coverage but was not implemented due to 
the additional computational cost. Some degree of semantic 
redundancy therefore likely remains in the final dataset.

In [19]:
# =============================================================================
# Task 1 Complete — Final File Verification and Statistics
# Reads final outputs and reports definitive dataset statistics
# including deduplication results and split counts.
# =============================================================================

# Load final deduplicated dataset for accurate counts
with open(ALL_QA_FILE, 'r', encoding='utf-8') as f:
    final_pairs = json.load(f)

# Load SQuAD splits for split counts
with open(SQUAD_FILE, 'r', encoding='utf-8') as f:
    squad_dataset = json.load(f)

def count_squad_pairs(squad_split):
    total        = 0
    answerable   = 0
    unanswerable = 0
    for article in squad_split['data']:
        for para in article['paragraphs']:
            for qa in para['qas']:
                total += 1
                if qa['is_impossible']:
                    unanswerable += 1
                else:
                    answerable += 1
    return total, answerable, unanswerable

tr_total, tr_ans, tr_unans = count_squad_pairs(squad_dataset['train'])
vl_total, vl_ans, vl_unans = count_squad_pairs(squad_dataset['validation'])
te_total, te_ans, te_unans = count_squad_pairs(squad_dataset['test'])
grand_total = tr_total + vl_total + te_total

n_answerable   = sum(1 for p in final_pairs if not p['is_impossible'])
n_unanswerable = sum(1 for p in final_pairs if p['is_impossible'])

# Output file verification
output_files = {
    "Corpus":        CORPUS_FILE,
    "Chunks":        CHUNKS_FILE,
    "QA pairs":      QA_PAIRS_FILE,
    "All QA pairs":  ALL_QA_FILE,
    "SQuAD dataset": SQUAD_FILE
}

print("Task 1 Complete — Output File Verification:")
print("-" * 60)
all_present = True
for name, filepath in output_files.items():
    if os.path.exists(filepath):
        size_kb = os.path.getsize(filepath) / 1024
        print(f"  ✅ {name:<20} {filepath:<30} ({size_kb:.0f} KB)")
    else:
        print(f"  ❌ {name:<20} {filepath:<30} (MISSING)")
        all_present = False

print("-" * 60)
if all_present:
    print("All output files present and ready for Task 2.")
else:
    print("WARNING: Some output files are missing.")

print(f"\nFinal Dataset Statistics:")
print(f"  Source documents downloaded:  74")
print(f"  Documents extracted:          71")
print(f"  Total corpus words:           625,400")
print(f"  Total chunks:                 2,109")
print(f"  Pairs before deduplication:   4,200")
print(f"  Pairs after deduplication:    {len(final_pairs):,}")
print(f"  Duplicates removed:           {4200 - len(final_pairs):,}")
print(f"  Answerable pairs:             {n_answerable:,}")
print(f"  Unanswerable pairs:           {n_unanswerable:,}")

print(f"\nSQuAD 2.0 Dataset Splits:")
print(f"{'Split':<15} {'Total':>8} {'Answerable':>12} "
      f"{'Unanswerable':>14} {'Ratio':>8}")
print(f"{'-'*60}")
print(f"{'Train':<15} {tr_total:>8,} {tr_ans:>12,} "
      f"{tr_unans:>14,} {tr_total/grand_total*100:>7.1f}%")
print(f"{'Validation':<15} {vl_total:>8,} {vl_ans:>12,} "
      f"{vl_unans:>14,} {vl_total/grand_total*100:>7.1f}%")
print(f"{'Test':<15} {te_total:>8,} {te_ans:>12,} "
      f"{te_unans:>14,} {te_total/grand_total*100:>7.1f}%")
print(f"{'-'*60}")
print(f"{'Total':<15} {grand_total:>8,} "
      f"{tr_ans+vl_ans+te_ans:>12,} "
      f"{tr_unans+vl_unans+te_unans:>14,}")

# Category breakdown
cat_pairs = {}
for pair in final_pairs:
    cat = pair['category']
    cat_pairs[cat] = cat_pairs.get(cat, 0) + 1

print(f"\nPairs by source category:")
for cat, count in sorted(cat_pairs.items()):
    pct = count / len(final_pairs) * 100
    print(f"  {cat:<15} {count:>6,} ({pct:.1f}%)")

Task 1 Complete — Output File Verification:
------------------------------------------------------------
  ✅ Corpus               ./corpus.json                  (4119 KB)
  ✅ Chunks               ./chunks.json                  (5949 KB)
  ✅ QA pairs             ./qa_pairs.json                (10762 KB)
  ✅ All QA pairs         ./all_qa_pairs.json            (10360 KB)
  ✅ SQuAD dataset        ./squad_dataset.json           (6661 KB)
------------------------------------------------------------
All output files present and ready for Task 2.

Final Dataset Statistics:
  Source documents downloaded:  74
  Documents extracted:          71
  Total corpus words:           625,400
  Total chunks:                 2,109
  Pairs before deduplication:   4,200
  Pairs after deduplication:    3,379
  Duplicates removed:           821
  Answerable pairs:             3,034
  Unanswerable pairs:           345

SQuAD 2.0 Dataset Splits:
Split              Total   Answerable   Unanswerable    Ratio
-----

### This completes Task 1